In [ ]:
# CANDIDATE BUILD: tuned for a financial-services data professional
# (data governance, data quality, data management, regulatory reporting, risk data,
#  MI/BI reporting, business analysis; analyst to specialist level, UK/London).
# OPENROUTER MANUAL REVIEW NOTE: the daily export is deterministic; the optional AI
# review uses OpenRouter via OPENROUTER_API_KEY and fills AI Remarks for a shortlist only.
# EXCEL NOTE: the Jobs sheet stores the FULL job description (never trimmed) and the
# Apply Link column is a real clickable hyperlink.
# CLEAN VERSION NOTE: Run all cells in Google Colab. Only the final export cell downloads Excel, with Jobs + Networking Tracker tabs.
!pip install requests feedparser pandas openpyxl beautifulsoup4 -q
print("âœ… Packages ready.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CANDIDATE PROFILE THIS NOTEBOOK IS TUNED FOR (from the CV only):
#   - Data Management Analyst at a US bank: FR Y-14 (Schedule H1/H2) submissions,
#     regulatory data reconciliation, data quality packages, BCBS 239 checks,
#     governance documentation and audit trails across Commercial Banking.
#   - Business Analyst at an analytics firm: data governance framework build,
#     metadata repositories, information standards, executive-level summaries,
#     stakeholder management on ML delivery.
#   - MSc Programme & Project Management (in progress); BTech Information Technology.
#   - SQL, Python, Advanced Excel/VBA, Power BI, Tableau, Alteryx, SharePoint,
#     ServiceNow, JIRA. APM certification, Power BI data analytics certification.
#   - ~4 years' post-graduate experience: analyst / senior analyst / associate /
#     specialist level. Manager, Head of and Director roles are NOT assumed.
# Everything below is derived from that and nothing else.
# ─────────────────────────────────────────────────────────────────────────────

import os


# Colab Secrets and environment variables take precedence over any value stored
# in this notebook, so keys can be rotated without editing the cell.
def _secret(name, default=""):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(name, default)


# ── CREDENTIALS ──────────────────────────────────────────────────────────────
# Two ways to supply the keys. Either works; Colab Secrets wins if both are set.
#
#   A) Paste them inline, between the quotes on the three lines below. Simplest,
#      and it travels with the file - so keep the file private, because anyone
#      who opens it can then use your keys.
#   B) Leave the quotes empty and add the same three names under Colab Secrets
#      (key icon in the left sidebar, then enable notebook access).
#
# LinkedIn needs no key at all - it is read from its public job-search pages.
ADZUNA_APP_ID  = _secret("ADZUNA_APP_ID",  "")   # <-- paste your Adzuna app id here
ADZUNA_APP_KEY = _secret("ADZUNA_APP_KEY", "")   # <-- paste your Adzuna app key here
REED_API_KEY   = _secret("REED_API_KEY",   "")   # <-- paste your Reed API key here

# Email settings are read the same way. Nothing in this notebook sends email
# today; these are kept only so existing references keep resolving.
SENDER_EMAIL    = _secret("SENDER_EMAIL")
SENDER_PASSWORD = _secret("SENDER_PASSWORD")
RECEIVER_EMAIL  = _secret("RECEIVER_EMAIL")

_missing = [name for name, value in [
    ("ADZUNA_APP_ID", ADZUNA_APP_ID),
    ("ADZUNA_APP_KEY", ADZUNA_APP_KEY),
    ("REED_API_KEY", REED_API_KEY),
] if not value]
if _missing:
    print("!! Missing credentials: " + ", ".join(_missing))
    print("   Paste them into the marked lines in this cell, or add them under")
    print("   Colab Secrets, then re-run this cell. Adzuna and Reed searches")
    print("   return nothing until they are set; the company career-page,")
    print("   Workday and LinkedIn sources need no key and work regardless.")
else:
    print("Credentials loaded for Adzuna and Reed.")

# ── SEARCH SETTINGS ──────────────────────────────────────────────────────────
LOCATION           = "London"
SALARY_MIN         = 35000
RESULTS_PER_SOURCE = 25

# Output workbook naming (candidate specific).
CANDIDATE_SLUG = "riya"
OUTPUT_FILE    = f"{CANDIDATE_SLUG}_jobs_final.xlsx"
AI_OUTPUT_FILE = f"{CANDIDATE_SLUG}_jobs_ai_review.xlsx"

# Internships/placements are OFF by default: the CV shows ~4 years of post-graduate
# experience, so graduate schemes and entry-level internships are a downgrade.
# Set to True to also search summer/industrial placements during the MSc year.
INCLUDE_INTERNSHIPS = False

# ── ROLE TAXONOMY ────────────────────────────────────────────────────────────
# PRIMARY: directly evidenced by the CV.
PRIMARY_TRACK_TERMS = {
    "Data Governance": [
        "data governance", "information governance", "data steward", "data stewardship",
        "data owner", "data policy", "data controls", "data control", "metadata management",
        "data lineage", "master data", "mdm", "reference data", "data catalogue", "data catalog",
    ],
    "Data Quality": [
        "data quality", "data assurance", "data integrity", "data validation",
        "data remediation", "data cleansing", "data profiling", "dq analyst",
    ],
    "Data Management": [
        "data management", "data operations", "data analyst", "data administration",
        "data services", "information management",
    ],
    "Regulatory Reporting": [
        "regulatory reporting", "regulatory report", "reg reporting", "prudential reporting",
        "financial reporting analyst", "statutory reporting", "corep", "finrep", "fr y-14",
        "fry-14", "ccar", "regulatory returns",
    ],
    "Regulatory Data": [
        "regulatory data", "regulatory change", "regulatory analyst", "compliance data",
        "regulatory controls", "regulatory operations",
    ],
    "Risk Data": [
        "risk data", "risk reporting", "credit risk analyst", "risk analytics",
        "risk control", "risk mi", "credit risk reporting",
    ],
    "Financial Data Analysis": [
        "financial data analyst", "finance data analyst", "financial data", "finance analyst",
        "financial analyst", "reconciliation analyst", "financial control analyst",
        "finance business analyst", "financial crime data",
    ],
    "Business Analysis": [
        "business analyst", "business data analyst", "data business analyst",
        "business analysis", "process analyst", "business process analyst",
    ],
    "MI / Reporting": [
        "mi analyst", "mi reporting", "management information", "reporting analyst",
        "reporting specialist", "insight analyst", "performance reporting",
    ],
    "Business Intelligence": [
        "business intelligence", "bi analyst", "bi developer", "power bi", "tableau",
        "bi reporting", "visualisation analyst", "visualization analyst",
    ],
    "Financial Services Analytics": [
        "financial services analyst", "banking analyst", "client data analyst",
        "kyc data", "controls analyst", "operations analyst",
    ],
}

# SECONDARY: adjacent, supported by the MSc plus delivery/documentation experience.
SECONDARY_TRACK_TERMS = {
    "PMO / Project": [
        "pmo analyst", "pmo", "project analyst", "programme analyst", "program analyst",
        "project support", "project coordinator", "project management officer",
        "portfolio analyst", "delivery analyst",
    ],
    "Change / Transformation": [
        "transformation analyst", "change analyst", "business change analyst",
        "change management analyst", "transformation associate", "business readiness",
    ],
    "Data / Technology Projects": [
        "data project", "data programme", "data program", "technology project",
        "technology business analyst", "data delivery", "data migration analyst",
    ],
    "Analytics": [
        "analytics analyst", "analytics associate", "data insights", "insights analyst",
        "reporting and analytics", "decision science analyst",
    ],
}

# STRETCH: plausible but a bigger jump; scored lower and flagged as a stretch.
STRETCH_TRACK_TERMS = {
    "Consulting": [
        "data consultant", "analytics consultant", "data governance consultant",
        "risk consultant", "regulatory consultant", "management consultant",
        "technology consultant", "transformation consultant", "business consultant",
    ],
    "Product / Strategy": [
        "product analyst", "product owner", "strategy analyst", "commercial analyst",
        "pricing analyst",
    ],
}

PRIMARY_TITLE_TERMS   = sorted({t for v in PRIMARY_TRACK_TERMS.values()   for t in v})
SECONDARY_TITLE_TERMS = sorted({t for v in SECONDARY_TRACK_TERMS.values() for t in v})
STRETCH_TITLE_TERMS   = sorted({t for v in STRETCH_TRACK_TERMS.values()   for t in v})

# Kept so any cell still reading TIER1/TIER2/TIER3 keeps working.
TIER1 = PRIMARY_TITLE_TERMS
TIER2 = SECONDARY_TITLE_TERMS
TIER3 = STRETCH_TITLE_TERMS

# ── SEARCH KEYWORDS ──────────────────────────────────────────────────────────
# High recall: exact titles, synonyms, abbreviations, banking/regulatory wording.
# Precision is applied later by the exclusion layer and score_job(), not here.
CORE_SEARCH_KEYWORDS = [
    # Data governance / quality / management
    "data governance analyst",
    "data governance specialist",
    "data quality analyst",
    "data quality specialist",
    "data management analyst",
    "data management specialist",
    "data controls analyst",
    "data steward",
    "metadata analyst",
    "master data analyst",
    "reference data analyst",
    "data analyst financial services",
    # Regulatory / risk
    "regulatory reporting analyst",
    "regulatory data analyst",
    "regulatory reporting specialist",
    "risk data analyst",
    "risk reporting analyst",
    "prudential regulatory reporting",
    "compliance data analyst",
    "financial crime data analyst",
    # Financial data / reporting
    "financial data analyst",
    "financial reporting analyst",
    "reconciliation analyst",
    "reporting analyst banking",
    "mi analyst",
    "management information analyst",
    # Business analysis / BI
    "business analyst data",
    "business data analyst",
    "business intelligence analyst",
    "bi analyst",
    "power bi analyst",
    "business process analyst",
    "analytics analyst",
    # Project / change (secondary)
    "data project analyst",
    "pmo analyst",
    "project analyst",
    "programme analyst",
    "transformation analyst",
    "change analyst",
    # Stretch
    "data governance consultant",
    "data analytics consultant",
    "risk and regulatory consultant",
]

# Only added to SEARCH_KEYWORDS when INCLUDE_INTERNSHIPS is True.
INTERNSHIP_KEYWORDS = [
    "data analyst placement",
    "data internship",
    "summer internship data",
    "industrial placement data",
]

SEARCH_KEYWORDS = CORE_SEARCH_KEYWORDS + (INTERNSHIP_KEYWORDS if INCLUDE_INTERNSHIPS else [])

# Seniority supported by the CV.
SUPPORTED_SENIORITY = [
    "analyst", "senior analyst", "associate", "specialist", "consultant",
    "officer", "executive", "advisor", "adviser", "coordinator", "administrator",
    "lead analyst",
]
# Seniority the CV does not support for these functions. Still collected, but
# scored down and flagged as a stretch rather than silently dropped.
UNSUPPORTED_SENIORITY = [
    "head of", "director", "vice president", " vp ", "vp,", "managing director",
    "chief ", "partner", "principal", "senior manager", "global head",
]

# Broad recall vocabulary (not a hard gate in the collection path).
RELEVANT_TITLES = [
    "data", "data governance", "data quality", "data management", "data steward",
    "metadata", "lineage", "master data", "reference data", "regulatory", "reporting",
    "regulatory reporting", "risk data", "risk reporting", "financial data", "finance",
    "reconciliation", "controls", "mi", "management information", "business intelligence",
    "bi", "power bi", "tableau", "analytics", "analysis", "business analyst",
    "business process", "governance", "compliance data", "pmo", "project", "programme",
    "program", "transformation", "change", "analyst", "associate", "specialist",
    "consultant",
]

# ── EXCLUSION LOGIC ──────────────────────────────────────────────────────────
# HARD: never relevant to this CV, whatever else the advert says.
HARD_EXCLUDE = [
    # Build-side technology
    "software engineer", "software developer", "backend engineer", "back end engineer",
    "frontend engineer", "front end engineer", "full stack", "fullstack",
    "data engineer", "analytics engineer", "platform engineer", "cloud engineer",
    "devops", "site reliability", "network engineer", "security engineer",
    "qa engineer", "test engineer", "automation engineer", "mobile engineer",
    "ios developer", "android developer", "java developer", ".net developer",
    "python developer", "solutions architect", "data architect",
    "machine learning engineer", "ml engineer", "ai engineer", "research scientist",
    "data scientist", "machine learning scientist",
    # Markets / quant
    "quantitative analyst", "quant analyst", "quantitative researcher", "quant researcher",
    "trader", "trading desk", "sales trader", "structuring", "market maker",
    # Finance qualifications the CV does not hold
    "accountant", "management accountant", "financial accountant", "bookkeeper",
    "tax analyst", "tax manager", "tax adviser", "tax advisor", "actuary", "actuarial",
    # Legal / clinical / trades / hospitality / retail / logistics / facilities
    "solicitor", "lawyer", "legal counsel", "paralegal", "barrister",
    "nurse", "nursing", "clinical", "care home", "healthcare assistant", "pharmacist",
    "chef", "barista", "restaurant", "hotel operations", "housekeeping", "catering",
    "hospitality", "retail store", "store manager", "shop manager", "sales assistant",
    "warehouse", "logistics operative", "driver", "hgv", "forklift", "courier",
    "electrician", "plumber", "installer", "field technician", "maintenance technician",
    "facilities manager", "facility manager", "estate manager", "site manager",
    "mechanical engineer", "civil engineer", "electrical engineer", "structural engineer",
    "teacher", "lecturer", "recruitment consultant", "recruiter", "telesales",
    "call centre", "call center", "customer service advisor", "cleaner", "security officer",
    # Materially junior entry routes
    "graduate scheme", "graduate programme", "graduate program", "apprentice",
    "apprenticeship", "school leaver", "work experience",
]

# SOFT: dropped only when nothing in the advert connects it to this CV, so
# "Compliance Data Analyst", "Regulatory Data Analyst" and "Audit Data Analyst"
# survive via RESCUE_TERMS below.
SOFT_EXCLUDE = [
    "internal audit", "external audit", "audit manager", "audit senior", "auditor",
    "compliance officer", "compliance manager", "compliance analyst", "aml analyst",
    "financial crime officer", "underwriter", "claims handler", "mortgage adviser",
    "marketing", "brand ", "procurement", "supply chain", "people partner",
    "payroll", "sales manager", "account manager", "business development manager",
]

# Any of these anywhere in title + description pulls a SOFT-excluded advert back in.
RESCUE_TERMS = [
    "data governance", "data quality", "data management", "data steward", "data lineage",
    "metadata", "master data", "reference data", "data controls", "regulatory reporting",
    "regulatory data", "risk data", "data analyst", "data analysis", "sql", "power bi",
    "tableau", "alteryx", "bcbs", "fr y-14", "fry-14", "corep", "finrep", "ccar",
    "reconciliation", "management information", "business intelligence", "data remediation",
]

# The collection cells use this as their title-level filter (via title_is_excluded).
EXCLUDE_TITLES = HARD_EXCLUDE + SOFT_EXCLUDE
if not INCLUDE_INTERNSHIPS:
    EXCLUDE_TITLES = EXCLUDE_TITLES + ["internship", "summer intern", "placement student"]

# Technical depth the CV does not evidence. Lowers the score and raises stretch;
# never removes the row on its own.
UNSUPPORTED_TECH_TERMS = [
    "spark", "scala", "hadoop", "kafka", "airflow", "dbt ", "databricks", "snowflake",
    "kubernetes", "terraform", "ci/cd", "microservices", "deep learning", "mlops",
    "feature engineering", "pytorch", "tensorflow", "golang", "react", "node.js",
]

# Software-requirements-engineering BA roles the CV does not support.
SOFTWARE_BA_TERMS = [
    "user stories", "acceptance criteria", "backlog grooming", "scrum ceremonies",
    "software development lifecycle", "sdlc", "api specification",
    "technical specification", "system design", "solution design", "wireframe",
]

# ── LOCATION ─────────────────────────────────────────────────────────────────
NON_UK = [
    "united states", " usa", " us,", "u.s.", "canada", "australia", "japan",
    "singapore", "india", "germany", "france", "spain", "italy",
    "mexico", "brazil", "new york", "san francisco", "seattle", "boston",
    "los angeles", "washington", "chicago", "dallas", "miami", "austin",
    "tokyo", "sydney", "melbourne", "toronto", "paris", "berlin", "barcelona",
    "rome", "gurugram", "gurgaon", "mumbai", "bangalore", "bengaluru",
    "munich", "muenchen", "m?nchen", "frankfurt", "dublin", "ireland",
    "portugal", "lisbon", "sweden", "stockholm", "netherlands", "belgium",
    "poland", "hyderabad", "delhi", "sofia", "riga", "warsaw", "amsterdam",
    "madrid", "milan", "shanghai", "beijing", "hong kong", "seoul", "dubai",
    "abu dhabi", "riyadh", "apac", "emea", "latam", "remote - us", "remote us",
]

UK_LOCATION_SIGNALS = [
    "london", "united kingdom", "uk", "england", "remote uk", "remote - uk",
    "hybrid - london", "manchester", "edinburgh", "birmingham", "bristol",
    "leeds", "sheffield", "cardiff", "glasgow", "coventry", "oxford", "cambridge",
    "belfast", "newcastle", "nottingham", "reading", "milton keynes", "wales",
    "scotland", "northern ireland",
]

print("Settings loaded: data governance / data quality / regulatory reporting profile.")
print(f"  Search keywords:    {len(SEARCH_KEYWORDS)}")
print(f"  Primary tracks:     {len(PRIMARY_TRACK_TERMS)} ({len(PRIMARY_TITLE_TERMS)} title terms)")
print(f"  Secondary tracks:   {len(SECONDARY_TRACK_TERMS)} ({len(SECONDARY_TITLE_TERMS)} title terms)")
print(f"  Stretch tracks:     {len(STRETCH_TRACK_TERMS)} ({len(STRETCH_TITLE_TERMS)} title terms)")
print(f"  Hard exclusions:    {len(HARD_EXCLUDE)} | Soft (rescuable): {len(SOFT_EXCLUDE)}")
print(f"  Internships:        {'ON' if INCLUDE_INTERNSHIPS else 'OFF'}")


In [ ]:
import re

LOCATION_MAP = {
    "london": ("UK", "Europe"), "united kingdom": ("UK", "Europe"), "england": ("UK", "Europe"),
    "remote uk": ("UK", "Europe"), "remote - uk": ("UK", "Europe"), "manchester": ("UK", "Europe"),
    "edinburgh": ("UK", "Europe"), "birmingham": ("UK", "Europe"), "bristol": ("UK", "Europe"),
    "leeds": ("UK", "Europe"), "sheffield": ("UK", "Europe"), "cardiff": ("UK", "Europe"),
    "glasgow": ("UK", "Europe"), "coventry": ("UK", "Europe"), "oxford": ("UK", "Europe"),
    "cambridge": ("UK", "Europe"),
    "amsterdam": ("Netherlands", "Europe"), "netherlands": ("Netherlands", "Europe"),
    "madrid": ("Spain", "Europe"), "spain": ("Spain", "Europe"), "barcelona": ("Spain", "Europe"),
    "paris": ("France", "Europe"), "france": ("France", "Europe"), "berlin": ("Germany", "Europe"),
    "germany": ("Germany", "Europe"), "munich": ("Germany", "Europe"), "muenchen": ("Germany", "Europe"),
    "frankfurt": ("Germany", "Europe"), "zurich": ("Switzerland", "Europe"), "switzerland": ("Switzerland", "Europe"),
    "dublin": ("Ireland", "Europe"), "ireland": ("Ireland", "Europe"), "milan": ("Italy", "Europe"),
    "rome": ("Italy", "Europe"), "italy": ("Italy", "Europe"), "lisbon": ("Portugal", "Europe"),
    "portugal": ("Portugal", "Europe"), "stockholm": ("Sweden", "Europe"), "sweden": ("Sweden", "Europe"),
    "warsaw": ("Poland", "Europe"), "poland": ("Poland", "Europe"), "sofia": ("Bulgaria", "Europe"),
    "riga": ("Latvia", "Europe"), "brussels": ("Belgium", "Europe"), "belgium": ("Belgium", "Europe"),
    "new york": ("USA", "North America"), "san francisco": ("USA", "North America"),
    "chicago": ("USA", "North America"), "seattle": ("USA", "North America"), "boston": ("USA", "North America"),
    "los angeles": ("USA", "North America"), "washington": ("USA", "North America"),
    "united states": ("USA", "North America"), "usa": ("USA", "North America"),
    "dallas": ("USA", "North America"), "miami": ("USA", "North America"), "austin": ("USA", "North America"),
    "toronto": ("Canada", "North America"), "canada": ("Canada", "North America"),
    "singapore": ("Singapore", "Asia Pacific"), "hong kong": ("Hong Kong", "Asia Pacific"),
    "tokyo": ("Japan", "Asia Pacific"), "japan": ("Japan", "Asia Pacific"),
    "sydney": ("Australia", "Asia Pacific"), "melbourne": ("Australia", "Asia Pacific"),
    "australia": ("Australia", "Asia Pacific"), "beijing": ("China", "Asia Pacific"),
    "shanghai": ("China", "Asia Pacific"), "china": ("China", "Asia Pacific"),
    "seoul": ("South Korea", "Asia Pacific"), "mumbai": ("India", "South Asia"),
    "delhi": ("India", "South Asia"), "gurugram": ("India", "South Asia"), "gurgaon": ("India", "South Asia"),
    "hyderabad": ("India", "South Asia"), "bangalore": ("India", "South Asia"), "bengaluru": ("India", "South Asia"),
    "india": ("India", "South Asia"), "dubai": ("UAE", "Middle East"), "abu dhabi": ("UAE", "Middle East"),
    "uae": ("UAE", "Middle East"), "riyadh": ("Saudi Arabia", "Middle East"), "saudi": ("Saudi Arabia", "Middle East"),
    "sao paulo": ("Brazil", "Latin America"), "brazil": ("Brazil", "Latin America"), "mexico": ("Mexico", "Latin America"),
}

NON_UK_COUNTRIES = {country for country, _ in LOCATION_MAP.values() if country != "UK"}


def normalise_text(value):
    return re.sub(r"\s+", " ", str(value or "").lower()).strip()


def get_country_continent(location):
    loc = normalise_text(location)
    if not loc or loc in {"see listing", "live now"}:
        return "Unknown", "Unknown"
    for keyword, result in LOCATION_MAP.items():
        if keyword in loc:
            return result
    if any(sig in loc for sig in UK_LOCATION_SIGNALS):
        return "UK", "Europe"
    return "Unknown", "Unknown"


def is_uk_loc(loc):
    loc_lower = normalise_text(loc)
    if not loc_lower:
        return True
    if any(non_uk in loc_lower for non_uk in NON_UK):
        return False
    country, _ = get_country_continent(loc_lower)
    if country in NON_UK_COUNTRIES:
        return False
    if country == "UK":
        return True
    # Unknown company-page locations are allowed into scoring, then reviewed by post-scoring filters.
    return True


def sponsorship_risk(text):
    t = normalise_text(text)
    no_sponsorship = [
        "no sponsorship", "cannot sponsor", "unable to sponsor", "will not sponsor",
        "does not sponsor", "do not sponsor", "must have right to work",
        "right to work in the uk", "eligible to work in the uk", "existing right to work",
        "without sponsorship", "sponsorship is not available",
    ]
    positive = ["visa sponsorship", "sponsorship available", "skilled worker", "sponsor licence", "graduate visa"]
    if any(phrase in t for phrase in no_sponsorship):
        return "High - right to work/no sponsorship wording"
    if any(phrase in t for phrase in positive):
        return "Low - sponsorship/visa friendly signal"
    return "Unknown"

# ── CANDIDATE-FIT VOCABULARY ─────────────────────────────────────────────────
# Every list below maps to something the CV actually evidences.
FUNCTIONAL_TERMS = [
    "data quality", "data governance", "data management", "data controls", "data steward",
    "data lineage", "metadata", "master data", "reference data", "regulatory reporting",
    "regulatory data", "reconciliation", "reconcile", "month-end", "month end",
    "management information", "mi reporting", "dashboard", "reporting pack",
    "data validation", "data remediation", "root cause", "issue resolution",
    "controls testing", "documentation", "audit trail", "governance framework",
    "process improvement", "requirements gathering", "uat", "data dictionary",
]

TECHNICAL_TERMS = [
    "sql", "python", "excel", "vba", "power bi", "powerbi", "tableau", "alteryx",
    "sharepoint", "servicenow", "jira", "power query", "pivot", "data visualisation",
    "data visualization", "reporting tools", "macros",
]

DOMAIN_TERMS = [
    "bank", "banking", "financial services", "capital markets", "asset management",
    "wealth management", "insurance", "payments", "fintech", "credit", "lending",
    "commercial banking", "retail banking", "investment bank", "treasury",
    "financial institution", "regtech", "buy-side", "sell-side",
]

REGULATORY_TERMS = [
    "bcbs 239", "bcbs239", "fr y-14", "fry-14", "fry14", "ccar", "corep", "finrep",
    "basel", "prudential", "pra ", "fca", "federal reserve", "regulatory requirement",
    "regulatory framework", "regulatory compliance", "gdpr", "dora", "mifid", "emir",
    "sox", "risk framework", "regulatory submission", "regulatory return",
]

TRANSFER_TERMS = [
    "stakeholder", "cross-functional", "senior stakeholders", "executive", "presentation",
    "communication", "project", "programme", "delivery", "governance", "planning",
    "prioritisation", "documentation", "workshops", "business case", "change",
    "continuous improvement", "problem solving", "attention to detail",
]

SENIORITY_POINTS = [
    ("senior analyst", 10), ("lead analyst", 9), ("senior associate", 9),
    ("analyst", 10), ("associate", 9), ("specialist", 9), ("officer", 8),
    ("coordinator", 7), ("administrator", 6), ("advisor", 7), ("adviser", 7),
    ("executive", 7), ("consultant", 7), ("senior consultant", 5),
    ("lead", 5), ("manager", 3), ("senior manager", 0), ("head of", 0),
    ("director", 0), ("vice president", 0), ("principal", 0), ("chief", 0),
]


def term_hits(text, terms):
    """Returns the subset of terms present in text."""
    return [term for term in terms if term in text]


def exclusion_reason(title, description=""):
    """
    Context-aware exclusion. HARD terms always exclude. SOFT terms exclude only
    when nothing in the advert ties it back to the candidate's data/regulatory
    background, so 'Compliance Data Analyst' or 'Audit Data Quality Analyst'
    survive while 'Compliance Officer' or 'Internal Auditor' do not.
    """
    t = normalise_text(title)
    d = normalise_text(description)
    combined = f"{t} {d}"

    hard = term_hits(t, HARD_EXCLUDE)
    if hard:
        return f"EXCLUDED - unsuitable role type ({hard[0]})"

    soft = term_hits(t, SOFT_EXCLUDE)
    if soft and not term_hits(combined, RESCUE_TERMS):
        return f"EXCLUDED - {soft[0]} with no data/regulatory content"

    return ""


def title_is_excluded(title, description=""):
    """Boolean wrapper used by the collection cells."""
    return bool(exclusion_reason(title, description))


# Titles made of words that appear on almost every advert. On their own they say
# nothing about fit, so they only score well when the advert body backs them up.
GENERIC_TITLE_TERMS = {
    "analyst", "data analyst", "business analyst", "business analysis",
    "analytics analyst", "reporting analyst", "insight analyst", "data insights",
    "operations analyst", "controls analyst", "project analyst", "delivery analyst",
    "commercial analyst", "strategy analyst", "product analyst", "product owner",
    "finance analyst", "financial analyst", "banking analyst", "pmo",
    "project support", "business consultant", "management consultant",
    "technology consultant", "process analyst",
}


def classify_track(title, description=""):
    """
    Returns (tier, track_name, track_points, matched_terms).
    A match in the title always beats a match found only in the description, and
    a primary track always beats a secondary or stretch one.
    """
    t = normalise_text(title)
    d = normalise_text(description)

    tiers = (
        ("Primary", PRIMARY_TRACK_TERMS, 32, 18),
        ("Secondary", SECONDARY_TRACK_TERMS, 22, 12),
        ("Stretch", STRETCH_TRACK_TERMS, 14, 7),
    )
    def best_match(haystack, group):
        """
        The most specific match wins. A meaningful term always beats a generic
        one, so 'Risk Data Analyst' maps to Risk Data rather than being swallowed
        by the generic 'data analyst'; ties are broken on term length.
        """
        candidates = []
        for track, terms in group.items():
            matched = [term for term in terms if term in haystack]
            if matched:
                specific = [term for term in matched if term not in GENERIC_TITLE_TERMS]
                rank = (1 if specific else 0, max(len(term) for term in (specific or matched)))
                candidates.append((rank, track, matched))
        if not candidates:
            return None
        _, track, matched = max(candidates, key=lambda x: x[0])
        return track, matched

    for tier, group, title_points, _ in tiers:
        found = best_match(t, group)
        if found:
            return tier, found[0], title_points, found[1]
    for tier, group, _, desc_points in tiers:
        found = best_match(d, group)
        if found:
            return f"{tier} (description only)", found[0], desc_points, found[1]
    return "Off-track", "Unmapped", 0, []


def extract_min_years(text):
    """Smallest 'N years' requirement mentioned in the advert, if any."""
    years = [int(y) for y in re.findall(r"(\d{1,2})\s*\+?\s*(?:years|yrs)", text)]
    years = [y for y in years if 0 < y <= 30]
    return min(years) if years else None


def seniority_points(title):
    t = normalise_text(title)
    matched = [(term, pts) for term, pts in SENIORITY_POINTS if term in t]
    if not matched:
        return 6, "unstated seniority"
    term, pts = min(matched, key=lambda x: x[1])
    return pts, term


def score_job(title, description="", location=""):
    """
    Candidate-fit scoring across ten dimensions. Keyword presence alone is not
    enough: a role only reaches the top buckets when the title sits on one of the
    candidate's tracks AND the advert shows the function, tooling, domain or
    regulatory content the CV evidences.

    Returns (total, track_points, reason, stretch, bucket) - same shape as before.
    """
    t = normalise_text(title)
    d = normalise_text(description)
    text = f"{t} {d}"

    reason_excluded = exclusion_reason(title, description)
    if reason_excluded:
        return 0, 0, reason_excluded, 10, "D - Skip"

    if location and not is_uk_loc(location):
        return 0, 0, f"Non-UK: {location}", 10, "D - Skip"

    # 1. Career-track relevance
    tier, track, track_score, track_terms = classify_track(title, description)

    # 2. Functional capability fit
    functional = term_hits(text, FUNCTIONAL_TERMS)
    functional_score = min(len(functional) * 2.5, 15)

    # 3. Technical skill fit
    technical = term_hits(text, TECHNICAL_TERMS)
    technical_score = min(len(technical) * 3, 12)

    # 4. Financial-services / domain fit
    domain = term_hits(text, DOMAIN_TERMS)
    domain_score = min(len(domain) * 2.5, 10)

    # 5. Regulatory / governance relevance
    regulatory = term_hits(text, REGULATORY_TERMS)
    regulatory_score = min(len(regulatory) * 3, 10)

    # A title made only of generic words ("Analyst", "Business Analyst", "Project
    # Analyst") must not score like a genuine match when nothing in the advert
    # supports it. A specific title such as "Risk Data Analyst" is left alone.
    supporting_signals = len(functional) + len(domain) + len(regulatory) + len(technical)
    generic_note = ""
    title_is_generic = bool(track_terms) and all(term in GENERIC_TITLE_TERMS for term in track_terms)
    if track_score >= 22 and title_is_generic and supporting_signals == 0:
        track_score = 12
        generic_note = " | generic title, no supporting content"

    # 6. Seniority fit
    seniority_score, seniority_term = seniority_points(title)
    if term_hits(t, UNSUPPORTED_SENIORITY):
        seniority_score = min(seniority_score, 2)

    # 7. Experience requirement fit
    min_years = extract_min_years(text)
    if min_years is None:
        experience_score = 4
        experience_note = "no stated experience bar"
    elif min_years <= 5:
        experience_score = 8
        experience_note = f"{min_years}+ yrs - within reach"
    elif min_years <= 7:
        experience_score = 2
        experience_note = f"{min_years}+ yrs - slightly above"
    else:
        experience_score = -8
        experience_note = f"{min_years}+ yrs - well above CV"

    # 8. Transferability
    transfer = term_hits(text, TRANSFER_TERMS)
    transfer_score = min(len(transfer) * 1.5, 8)

    # 9. Location fit
    location_score = 5 if (not location or any(sig in normalise_text(location) for sig in UK_LOCATION_SIGNALS)) else 2

    # 10. Overall application relevance adjustments
    penalties = []
    unsupported_tech = term_hits(text, UNSUPPORTED_TECH_TERMS)
    if len(unsupported_tech) >= 2:
        penalties.append(("heavy unsupported tech stack", -10))
    elif unsupported_tech:
        penalties.append(("some unsupported tech", -4))

    software_ba = term_hits(text, SOFTWARE_BA_TERMS)
    if "business analyst" in t and len(software_ba) >= 2 and not (functional or regulatory):
        penalties.append(("software-requirements BA, not data BA", -12))

    if tier == "Off-track":
        penalties.append(("no mapped career track", -6))

    penalty_total = sum(points for _, points in penalties)

    total = (
        track_score + functional_score + technical_score + domain_score
        + regulatory_score + seniority_score + experience_score
        + transfer_score + location_score + penalty_total
    )

    # Off-track roles never reach the apply buckets on generic keywords alone.
    if tier == "Off-track":
        total = min(total, 35)

    total = int(max(0, min(round(total), 100)))

    stretch = calculate_stretch(title, description, "", total, track_score, transfer_score)
    bucket = make_bucket_inline(total, stretch)

    reason = (
        f"{tier}: {track}{generic_note} | track {int(track_score)}, function {functional_score:g}, "
        f"tech {technical_score:g}, FS domain {domain_score:g}, regulatory {regulatory_score:g}, "
        f"seniority {seniority_score} ({seniority_term}), experience {experience_score:+d} ({experience_note}), "
        f"transferable {transfer_score:g}, location {location_score}"
    )
    if penalties:
        reason += " | " + "; ".join(f"{label} {points}" for label, points in penalties)
    if functional:
        reason += f" | matched: {', '.join(functional[:6])}"

    return total, int(track_score), reason, stretch, bucket


def make_bucket_inline(score, stretch):
    if score >= 70 and stretch <= 5:
        return "A - Apply Now"
    if score >= 60 and stretch <= 7:
        return "B - High Upside"
    if score >= 50 and stretch <= 9:
        return "C - Network First"
    return "D - Skip"


def calculate_stretch(title, description, company, score, interview_score=0, transfer_score=0):
    """
    1 = comfortably within reach for ~4 years' FS data experience at analyst level.
    10 = a major career pivot or a seniority the CV does not support.
    """
    t = normalise_text(title)
    text = f"{t} {normalise_text(description)} {normalise_text(company)}"
    stretch = 3

    if term_hits(t, UNSUPPORTED_SENIORITY):
        stretch += 3
    elif "manager" in t or "lead " in t:
        stretch += 2

    min_years = extract_min_years(text)
    if min_years is not None:
        if min_years >= 8:
            stretch += 3
        elif min_years >= 6:
            stretch += 1

    if len(term_hits(text, UNSUPPORTED_TECH_TERMS)) >= 2:
        stretch += 2

    if any(kw in text for kw in ["cfa", "aca ", "acca", "cima", "frm", "chartered accountant", "qualified accountant"]):
        stretch += 2

    if any(kw in text for kw in ["mckinsey", "bain", "bcg", "oliver wyman", "kearney", "goldman sachs", "blackstone"]):
        stretch += 2

    if any(term in t for term in SUPPORTED_SENIORITY):
        stretch = max(stretch - 1, 1)
    if any(term in t for term in PRIMARY_TITLE_TERMS):
        stretch = max(stretch - 1, 1)
    if interview_score >= 28 and transfer_score >= 5:
        stretch = max(stretch - 1, 1)
    if score >= 75:
        stretch = max(stretch - 1, 1)
    elif score < 50:
        stretch += 1

    return max(1, min(stretch, 10))


print("Scoring engine ready: 10-dimension candidate fit, context-aware exclusions,")
print("UK location filtering, stretch rating and sponsorship helpers.")


In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time

def fetch_adzuna(keyword):
    jobs = []
    url = (
        f"https://api.adzuna.com/v1/api/jobs/gb/search/1"
        f"?app_id={ADZUNA_APP_ID}&app_key={ADZUNA_APP_KEY}"
        f"&results_per_page={RESULTS_PER_SOURCE}"
        f"&what={requests.utils.quote(keyword)}"
        f"&where={requests.utils.quote(LOCATION)}"
        f"&salary_min={SALARY_MIN}&sort_by=date"
        f"&content-type=application/json"
    )
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        for item in r.json().get("results", []):
            jobs.append({
                "title":   item.get("title", ""),
                "company": item.get("company", {}).get("display_name", ""),
                "location": item.get("location", {}).get("display_name", ""),
                "salary":  f"{item.get('salary_min','')}â€“{item.get('salary_max','')}",
                "posted":  item.get("created", "")[:10],
                "desc":    item.get("description", ""),
                "url":     item.get("redirect_url", ""),
                "source":  "Adzuna",
            })
    except Exception as e:
        print(f"  Adzuna error [{keyword}]: {e}")
    return jobs

def fetch_reed(keyword):
    jobs = []
    url = (
        f"https://www.reed.co.uk/api/1.0/search"
        f"?keywords={requests.utils.quote(keyword)}"
        f"&locationName={requests.utils.quote(LOCATION)}"
        f"&minimumSalary={SALARY_MIN}"
        f"&resultsToTake={RESULTS_PER_SOURCE}"
    )
    try:
        r = requests.get(url, auth=(REED_API_KEY, ""), timeout=10)
        r.raise_for_status()
        for item in r.json().get("results", []):
            jobs.append({
                "title":   item.get("jobTitle", ""),
                "company": item.get("employerName", ""),
                "location": item.get("locationName", ""),
                "salary":  f"{item.get('minimumSalary','')}â€“{item.get('maximumSalary','')}",
                "posted":  item.get("date", "")[:10],
                "desc":    item.get("jobDescription", ""),
                "url":     item.get("jobUrl", ""),
                "source":  "Reed",
            })
    except Exception as e:
        print(f"  Reed error [{keyword}]: {e}")
    return jobs

print("Searching Adzuna + Reed...")
print("-" * 55)
board_jobs = []
for kw in SEARCH_KEYWORDS:
    a = fetch_adzuna(kw)
    r = fetch_reed(kw)
    board_jobs += a + r
    print(f"  {kw[:45]:<45} A:{len(a)} R:{len(r)}")
print("-" * 55)
print(f"âœ… Job boards: {len(board_jobs)}")

In [ ]:
import time
from bs4 import BeautifulSoup

headers_req = {"User-Agent": "Mozilla/5.0"}

# Cleans API-provided HTML descriptions while preserving plain text.
def clean_description(raw):
    if not raw:
        return ""
    return BeautifulSoup(raw, "html.parser").get_text(" ", strip=True)

company_jobs = []

GREENHOUSE = {
    # Fintech / challenger banks
    "Monzo":            "monzo",
    "GoCardless":       "gocardless",
    "Capital on Tap":   "capitalontap",
    "Adyen":            "adyen",
    "Stripe":           "stripe",
    "Marqeta":          "marqeta",
    "Dojo":             "dojo",
    "ComplyAdvantage":  "complyadvantage",
    "Tide":             "tide",
    "SumUp":            "sumup",
    "Liberis":          "liberis",
    "TrueLayer":        "truelayer",
    "Onfido":           "onfido",
    "Quantexa":         "quantexa",
    "Airbnb":           "airbnb",
    "Deliveroo":        "deliveroo",
    "Plaid":            "plaid",
    "Funding Circle":   "fundingcircle",
    "Freetrade":        "freetrade",
    "Marshmallow":      "marshmallow",
    "Iwoca":            "iwoca",
    "Nutmeg":           "nutmeg",
    "Zopa":             "zopabank",
    "Experian":         "experian",
    # Consulting
    "AlixPartners":     "alixpartners",
    "PA Consulting":    "paconsulting",
    "Capgemini":        "capgemini",
    "Slalom":           "slalom",
    "North Highland":   "northhighland",
    # Other
    "Octopus Energy GH": "octoenergy",
}

LEVER = {
    "Octopus Energy":   "octoenergy",
    "Moneybox":         "moneyboxapp",
    "OakNorth":         "oaknorth.ai",
    "Allica Bank":      "allica-bank",
    "Zego":             "zego",
    "TrueLayer LV":     "truelayer",
}

DIRECT = {
    "Deloitte":     "https://apply.deloitte.com/careers/SearchJobs/strategy",
    "EY Parthenon": "https://careers.ey.com/ey/search/?q=strategy+operations",
    "Bain":         "https://www.bain.com/careers/find-a-role/",
    "Barclays":     "https://search.jobs.barclays/search-jobs/London/22160/4",
    "HSBC":         "https://mycareer.hsbc.com/en_GB/external/SearchJobs",
    "Accenture":    "https://www.accenture.com/gb-en/careers/jobsearch",
    "BlackRock":    "https://careers.blackrock.com/",
    "Fidelity":     "https://careers.fidelityinternational.com/",
    "Schroders":    "https://www.schroders.com/en-gb/uk/individual/careers/",
}

NON_UK = [
    "united states", " us,", "canada", "australia", "japan",
    "singapore", "india", "germany", "france", "spain", "italy",
    "mexico", "brazil", "new york", "san francisco", "seattle",
    "tokyo", "sydney", "toronto", "paris", "berlin", "barcelona",
    "rome", "gurugram", "munich", "mÃ¼nchen", "dublin", "ireland",
    "portugal", "sweden", "netherlands", "belgium", "poland",
    "hyderabad", "delhi", "sofia", "riga", "warsaw", "amsterdam",
    "madrid", "milan", "shanghai", "beijing", "chicago",
]

def is_uk_loc(loc):
    if not loc: return True
    return not any(x in loc.lower() for x in NON_UK)

print("Checking company pages...")
print("-" * 55)

for company, token in GREENHOUSE.items():
    for base_url in [
        f"https://boards-api.greenhouse.io/v1/boards/{token}/jobs?content=true",
        f"https://boards.greenhouse.io/{token}",
    ]:
        try:
            r = requests.get(base_url, headers=headers_req, timeout=12)
            r.raise_for_status()
            found = 0
            if "api" in base_url:
                for job in r.json().get("jobs", []):
                    title = job.get("title", "")
                    offices = job.get("offices", [])
                    location = offices[0].get("name", "") if offices else ""
                    if not is_uk_loc(location): continue
                    # High recall: do not require title keyword match before AI review
                    if title_is_excluded(title): continue
                    company_jobs.append({
                        "title": title, "company": company,
                        "location": location or "UK",
                        "salary": "See listing", "posted": "Live now",
                        "desc": clean_description(job.get("content", "")) or title,
                        "description_available": "Yes" if job.get("content") else "No",
                        "description_quality": "Full API" if job.get("content") else "Title Only",
                        "source": "Career Page",
                        "url": job.get("absolute_url", ""),
                    })
                    found += 1
            else:
                soup = BeautifulSoup(r.text, "html.parser")
                for a in soup.find_all("a", href=True):
                    title = a.get_text(strip=True)
                    href = a["href"]
                    if len(title) < 8 or len(title) > 110: continue
                    # High recall: do not require title keyword match before AI review
                    if title_is_excluded(title): continue
                    full_url = href if href.startswith("http") else f"https://boards.greenhouse.io{href}"
                    company_jobs.append({
                        "title": title, "company": company,
                        "location": "UK", "salary": "See listing",
                        "posted": "Live now", "desc": title,
                        "source": "Career Page", "url": full_url,
                    })
                    found += 1
            print(f"  {company:<25} {found} roles")
            break
        except:
            continue
    time.sleep(0.3)

for company, slug in LEVER.items():
    url = f"https://api.lever.co/v0/postings/{slug}?mode=json"
    try:
        r = requests.get(url, headers=headers_req, timeout=12)
        r.raise_for_status()
        found = 0
        for job in r.json():
            title = job.get("text", "")
            location = job.get("categories", {}).get("location", "")
            if not is_uk_loc(location): continue
            # High recall: do not require title keyword match before AI review
            if title_is_excluded(title): continue
            company_jobs.append({
                "title": title, "company": company,
                "location": location or "UK",
                "salary": "See listing", "posted": "Live now",
                "desc": clean_description(job.get("descriptionPlain") or job.get("description") or "") or title,
                "description_available": "Yes" if (job.get("descriptionPlain") or job.get("description")) else "No",
                "description_quality": "Full API" if (job.get("descriptionPlain") or job.get("description")) else "Title Only",
                "source": "Career Page",
                "url": job.get("hostedUrl", ""),
            })
            found += 1
        print(f"  {company:<25} {found} roles")
    except:
        print(f"  {company:<25} could not reach")
    time.sleep(0.3)

for company, url in DIRECT.items():
    try:
        r = requests.get(url, headers=headers_req, timeout=15)
        soup = BeautifulSoup(r.text, "html.parser")
        found = 0
        for a in soup.find_all("a", href=True):
            title = a.get_text(strip=True)
            href = a["href"]
            if len(title) < 8 or len(title) > 110: continue
            # High recall: do not require title keyword match before AI review
            if title_is_excluded(title): continue
            full_url = href if href.startswith("http") else url.split("/")[0] + "//" + url.split("/")[2] + href
            company_jobs.append({
                "title": title, "company": company,
                "location": "London/UK", "salary": "See listing",
                "posted": "Live now", "desc": title,
                "source": "Career Page", "url": full_url,
            })
            found += 1
        print(f"  {company:<25} {found} roles")
    except:
        print(f"  {company:<25} could not reach")
    time.sleep(0.5)

# (Removed: hardcoded placeholder roles from the previous candidate's build.)

print("-" * 55)
print(f"âœ… Total company jobs: {len(company_jobs)}")


In [ ]:
# Cleans API-provided HTML descriptions while preserving plain text.
def clean_description(raw):
    if not raw:
        return ""
    return BeautifulSoup(raw, "html.parser").get_text(" ", strip=True)

# Add missing companies that were in the bigger list
GREENHOUSE_EXTRA = {
    # Data, regtech and financial-data employers added for this profile.
    "Quantexa DG":      "quantexa",
    "ComplyAdvantage DG": "complyadvantage",
    "Thought Machine":  "thoughtmachine",
    "Starling Bank":    "starlingbank",
    "FNZ":              "fnz",
    "Clearbank":        "clearbank",
    "Onfido":           "onfido",
    "Quantexa":         "quantexa",
    "Plaid":            "plaid",
    "Funding Circle":   "fundingcircle",
    "Freetrade":        "freetrade",
    "Marshmallow":      "marshmallow",
    "Iwoca":            "iwoca",
    "Nutmeg":           "nutmeg",
    "Experian":         "experian",
    "PA Consulting":    "paconsulting",
    "Capgemini":        "capgemini",
    "Slalom":           "slalom",
    "North Highland":   "northhighland",
    "Zopa":             "zopabank",
    "Checkout.com":     "checkoutdotcom",
    "Klarna":           "klarna",
    "Pleo":             "pleo",
    "Dojo Extra":       "dojo",
}

LEVER_EXTRA = {
    "Zego":             "zego",
    "Allica Bank":      "allica-bank",
    "Cleo":             "meetcleo",
    "Teneo":            "teneo",
}

print("Adding missing companies...")
print("-" * 55)
extra = 0

for company, token in GREENHOUSE_EXTRA.items():
    for base_url in [
        f"https://boards-api.greenhouse.io/v1/boards/{token}/jobs?content=true",
        f"https://boards.greenhouse.io/{token}",
    ]:
        try:
            r = requests.get(base_url, headers=headers_req, timeout=12)
            r.raise_for_status()
            found = 0
            if "api" in base_url:
                for job in r.json().get("jobs", []):
                    title = job.get("title", "")
                    offices = job.get("offices", [])
                    location = offices[0].get("name", "") if offices else ""
                    if not is_uk_loc(location): continue
                    # High recall: do not require title keyword match before AI review
                    if title_is_excluded(title): continue
                    company_jobs.append({
                        "title": title, "company": company,
                        "location": location or "UK",
                        "salary": "See listing", "posted": "Live now",
                        "desc": clean_description(job.get("content", "")) or title,
                        "description_available": "Yes" if job.get("content") else "No",
                        "description_quality": "Full API" if job.get("content") else "Title Only",
                        "source": "Career Page",
                        "url": job.get("absolute_url", ""),
                    })
                    found += 1
            else:
                soup = BeautifulSoup(r.text, "html.parser")
                for a in soup.find_all("a", href=True):
                    title = a.get_text(strip=True)
                    href = a["href"]
                    if len(title) < 8 or len(title) > 110: continue
                    # High recall: do not require title keyword match before AI review
                    if title_is_excluded(title): continue
                    full_url = href if href.startswith("http") else f"https://boards.greenhouse.io{href}"
                    company_jobs.append({
                        "title": title, "company": company,
                        "location": "UK", "salary": "See listing",
                        "posted": "Live now", "desc": title,
                        "source": "Career Page", "url": full_url,
                    })
                    found += 1
            print(f"  {company:<25} {found} roles")
            extra += found
            break
        except:
            continue
    time.sleep(0.3)

for company, slug in LEVER_EXTRA.items():
    url = f"https://api.lever.co/v0/postings/{slug}?mode=json"
    try:
        r = requests.get(url, headers=headers_req, timeout=12)
        r.raise_for_status()
        found = 0
        for job in r.json():
            title = job.get("text", "")
            location = job.get("categories", {}).get("location", "")
            if not is_uk_loc(location): continue
            # High recall: do not require title keyword match before AI review
            if title_is_excluded(title): continue
            company_jobs.append({
                "title": title, "company": company,
                "location": location or "UK",
                "salary": "See listing", "posted": "Live now",
                "desc": clean_description(job.get("descriptionPlain") or job.get("description") or "") or title,
                "description_available": "Yes" if (job.get("descriptionPlain") or job.get("description")) else "No",
                "description_quality": "Full API" if (job.get("descriptionPlain") or job.get("description")) else "Title Only",
                "source": "Career Page",
                "url": job.get("hostedUrl", ""),
            })
            found += 1
        print(f"  {company:<25} {found} roles")
        extra += found
    except:
        print(f"  {company:<25} could not reach")
    time.sleep(0.3)

print("-" * 55)
print(f"Extra roles added:    {extra}")
print(f"Total company jobs:   {len(company_jobs)}")


In [ ]:
# Old duplicate all-in-one export cell disabled.
# The notebook now uses the main fetch cells plus the final export cell only.
print("Skipped old duplicate export cell. Final Excel download happens in the last scoring/export cell.")


In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from google.colab import files
import time
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime

headers_req = {"User-Agent": "Mozilla/5.0"}

# Cleans API-provided HTML descriptions while preserving plain text.
def clean_description(raw):
    if not raw:
        return ""
    return BeautifulSoup(raw, "html.parser").get_text(" ", strip=True)



def request_with_backoff(url, retries=3, base_sleep=1.0, **kwargs):
    for attempt in range(retries):
        try:
            response = requests.get(url, **kwargs)
            if response.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"Retryable status {response.status_code}")
            return response
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(base_sleep * (2 ** attempt))


# â”€â”€ Company pages â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Keep anything the earlier company cells already collected instead of
# discarding it; duplicates are removed by URL in the final export cell.
company_jobs = list(globals().get("company_jobs", []))

GREENHOUSE = {
    "Monzo":            "monzo",
    "GoCardless":       "gocardless",
    "Capital on Tap":   "capitalontap",
    "Adyen":            "adyen",
    "Stripe":           "stripe",
    "Marqeta":          "marqeta",
    "Dojo":             "dojo",
    "ComplyAdvantage":  "complyadvantage",
    "Tide":             "tide",
    "SumUp":            "sumup",
    "Liberis":          "liberis",
    "TrueLayer":        "truelayer",
    "Onfido":           "onfido",
    "Airbnb":           "airbnb",
    "AlixPartners":     "alixpartners",
    "Funding Circle":   "fundingcircle",
    "Freetrade":        "freetrade",
    "Marshmallow":      "marshmallow",
    "Iwoca":            "iwoca",
    "Zopa":             "zopabank",
    "Plaid":            "plaid",
    "Experian":         "experian",
    "Checkout.com":     "checkoutdotcom",
    "Klarna":           "klarna",
    "Pleo":             "pleo",
    "Capgemini":        "capgemini",
    "Slalom":           "slalom",
}

LEVER = {
    "Octopus Energy":   "octoenergy",
    "Moneybox":         "moneyboxapp",
    "OakNorth":         "oaknorth.ai",
    "Zego":             "zego",
    "Cleo":             "meetcleo",
    "Teneo":            "teneo",
}

DIRECT = {
    "LSEG":         "https://www.lseg.com/en/careers/open-roles",
    "Moody's":      "https://careers.moodys.com/jobs",
    "S&P Global":   "https://careers.spglobal.com/jobs",
    "Grant Thornton": "https://www.grantthornton.co.uk/careers/",
    "NatWest":      "https://jobs.natwestgroup.com/search-jobs",
    "Deloitte":     "https://apply.deloitte.com/careers/SearchJobs/strategy",
    "EY Parthenon": "https://careers.ey.com/ey/search/?q=strategy+operations",
    "Bain":         "https://www.bain.com/careers/find-a-role/",
    "Barclays":     "https://search.jobs.barclays/search-jobs/London/22160/4",
    "HSBC":         "https://mycareer.hsbc.com/en_GB/external/SearchJobs",
    "Accenture":    "https://www.accenture.com/gb-en/careers/jobsearch",
    "BlackRock":    "https://careers.blackrock.com/",
    "Fidelity":     "https://careers.fidelityinternational.com/",
}

NON_UK = [
    "united states", " us,", "canada", "australia", "japan",
    "singapore", "india", "germany", "france", "spain", "italy",
    "mexico", "brazil", "new york", "san francisco", "seattle",
    "tokyo", "sydney", "toronto", "paris", "berlin", "barcelona",
    "rome", "gurugram", "munich", "mÃ¼nchen", "dublin", "ireland",
    "portugal", "sweden", "netherlands", "belgium", "poland",
    "hyderabad", "delhi", "sofia", "riga", "warsaw", "amsterdam",
    "madrid", "milan", "shanghai", "beijing", "chicago",
]

def is_uk_loc(loc):
    if not loc: return True
    return not any(x in loc.lower() for x in NON_UK)

print("Checking company pages...")
print("-" * 55)

for company, token in GREENHOUSE.items():
    for base_url in [
        f"https://boards-api.greenhouse.io/v1/boards/{token}/jobs?content=true",
        f"https://boards.greenhouse.io/{token}",
    ]:
        try:
            r = request_with_backoff(base_url, headers=headers_req, timeout=12)
            r.raise_for_status()
            found = 0
            if "api" in base_url:
                for job in r.json().get("jobs", []):
                    title = job.get("title", "")
                    offices = job.get("offices", [])
                    location = offices[0].get("name", "") if offices else ""
                    if not is_uk_loc(location): continue
                    # High recall: do not require title keyword match before AI review
                    if title_is_excluded(title): continue
                    company_jobs.append({
                        "title": title, "company": company,
                        "location": location or "UK",
                        "salary": "See listing", "posted": "Live now",
                        "desc": clean_description(job.get("content", "")) or title,
                        "description_available": "Yes" if job.get("content") else "No",
                        "description_quality": "Full API" if job.get("content") else "Title Only",
                        "source": "Career Page",
                        "url": job.get("absolute_url", ""),
                    })
                    found += 1
            else:
                soup = BeautifulSoup(r.text, "html.parser")
                for a in soup.find_all("a", href=True):
                    title = a.get_text(strip=True)
                    href = a["href"]
                    if len(title) < 8 or len(title) > 110: continue
                    # High recall: do not require title keyword match before AI review
                    if title_is_excluded(title): continue
                    full_url = href if href.startswith("http") else f"https://boards.greenhouse.io{href}"
                    company_jobs.append({
                        "title": title, "company": company,
                        "location": "UK", "salary": "See listing",
                        "posted": "Live now", "desc": title,
                        "source": "Career Page", "url": full_url,
                    })
                    found += 1
            print(f"  âœ… {company:<25} {found} roles")
            break
        except:
            continue
    time.sleep(0.3)

for company, slug in LEVER.items():
    url = f"https://api.lever.co/v0/postings/{slug}?mode=json"
    try:
        r = request_with_backoff(url, headers=headers_req, timeout=12)
        r.raise_for_status()
        found = 0
        for job in r.json():
            title = job.get("text", "")
            location = job.get("categories", {}).get("location", "")
            if not is_uk_loc(location): continue
            # High recall: do not require title keyword match before AI review
            if title_is_excluded(title): continue
            company_jobs.append({
                "title": title, "company": company,
                "location": location or "UK",
                "salary": "See listing", "posted": "Live now",
                "desc": clean_description(job.get("descriptionPlain") or job.get("description") or "") or title,
                "description_available": "Yes" if (job.get("descriptionPlain") or job.get("description")) else "No",
                "description_quality": "Full API" if (job.get("descriptionPlain") or job.get("description")) else "Title Only",
                "source": "Career Page",
                "url": job.get("hostedUrl", ""),
            })
            found += 1
        print(f"  âœ… {company:<25} {found} roles")
    except:
        print(f"  âŒ {company:<25} could not reach")
    time.sleep(0.3)

for company, url in DIRECT.items():
    try:
        r = request_with_backoff(url, headers=headers_req, timeout=15)
        soup = BeautifulSoup(r.text, "html.parser")
        found = 0
        for a in soup.find_all("a", href=True):
            title = a.get_text(strip=True)
            href = a["href"]
            if len(title) < 8 or len(title) > 110: continue
            # High recall: do not require title keyword match before AI review
            if title_is_excluded(title): continue
            full_url = href if href.startswith("http") else url.split("/")[0] + "//" + url.split("/")[2] + href
            company_jobs.append({
                "title": title, "company": company,
                "location": "London/UK", "salary": "See listing",
                "posted": "Live now", "desc": title,
                "source": "Career Page", "url": full_url,
            })
            found += 1
        print(f"  âœ… {company:<25} {found} roles")
    except:
        print(f"  âŒ {company:<25} could not reach")
    time.sleep(0.5)

# (Removed: hardcoded placeholder roles from the previous candidate's build.)
print(f"âœ… Total company jobs: {len(company_jobs)}")

# â”€â”€ LinkedIn â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
HEADERS_LI = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
linkedin_jobs = []
seen_li = set()
LI_SEARCHES = [
    "data+governance+analyst", "data+quality+analyst",
    "data+management+analyst", "data+steward",
    "regulatory+reporting+analyst", "regulatory+data+analyst",
    "risk+data+analyst", "financial+data+analyst",
    "reconciliation+analyst", "mi+analyst",
    "management+information+analyst", "reporting+analyst+banking",
    "business+data+analyst", "business+intelligence+analyst",
    "power+bi+analyst", "business+process+analyst",
    "data+project+analyst", "pmo+analyst",
    "transformation+analyst", "change+analyst",
    "data+governance+consultant",
]
print()
print("Pulling LinkedIn...")
print("-" * 55)
for kw in LI_SEARCHES:
    url = (f"https://www.linkedin.com/jobs/search/?keywords={kw}"
           f"&location=London%2C+United+Kingdom&f_TPR=r604800&f_E=2%2C3%2C4")
    try:
        r = request_with_backoff(url, headers=HEADERS_LI, timeout=15)
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.find_all("div", class_=lambda x: x and "base-card" in str(x))
        found = 0
        for card in cards:
            title_tag = card.find("h3", class_=lambda x: x and "base-search-card__title" in str(x))
            title = title_tag.get_text(strip=True) if title_tag else ""
            company_tag = card.find("h4", class_=lambda x: x and "base-search-card__subtitle" in str(x))
            company = company_tag.get_text(strip=True) if company_tag else ""
            loc_tag = card.find("span", class_=lambda x: x and "job-search-card__location" in str(x))
            location = loc_tag.get_text(strip=True) if loc_tag else "London"
            link_tag = card.find("a", class_=lambda x: x and "base-card__full-link" in str(x))
            link = link_tag["href"].split("?")[0] if link_tag else ""
            if not title or not company or not link or link in seen_li: continue
            # High recall: do not require title keyword match before AI review
            if title_is_excluded(title): continue
            seen_li.add(link)
            linkedin_jobs.append({
                "title": title, "company": company, "location": location,
                "salary": "See listing", "posted": "This week",
                "desc": title, "source": "LinkedIn", "url": link,
            })
            found += 1
        print(f"  {kw.replace('+', ' ')[:40]:<40} {found} roles")
        time.sleep(2)
    except:
        print(f"  {kw[:40]} blocked")
print(f"âœ… LinkedIn: {len(linkedin_jobs)} roles")

# â”€â”€ Score â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print()
print("Scoring all jobs...")
results = []
seen_urls = set()

MISMATCH_RULES = [
    {"companies": ["jp morgan", "jpmorgan", "goldman sachs", "morgan stanley",
                   "deutsche bank", "ubs", "citi", "bank of america",
                   "bnp paribas", "lazard", "rothschild", "evercore"],
     "titles": ["vice president", " vp ", "managing director", "cib",
                "quantitative", "structuring", "trading"]},
    {"companies": ["skanska", "bovis", "mace", "wsp", "aecom",
                   "balfour", "costain", "kier"],
     "titles": ["commercial manager", "senior commercial manager"]},
]

def fails_mismatch(title, company):
    t = title.lower()
    c = company.lower()
    for rule in MISMATCH_RULES:
        if any(kw in c for kw in rule["companies"]) and any(kw in t for kw in rule["titles"]):
            return True
    return False

for job in board_jobs + company_jobs + linkedin_jobs:
    url = job.get("url", "")
    if not url or url in seen_urls: continue
    seen_urls.add(url)
    if fails_mismatch(job["title"], job.get("company", "")): continue
    total, interview_prob, reason, stretch, bucket = score_job(
        job["title"], job.get("desc", ""), job.get("location", "")
    )
    if job.get("source") == "Career Page" and total >= 25:
        total = min(total + 20, 100)
        reason = reason + " | +20 direct boost"
    if total < 40: continue
    results.append({
        "Bucket": "TBC", "Score /100": total, "Stretch (1-10)": stretch,
        "Title": job["title"], "Company": job["company"],
        "Location": job.get("location", ""),
        "Salary": job.get("salary", "See listing"),
        "Posted": job.get("posted", ""), "Source": job["source"],
        "Why It Matches": reason, "Status": "To Review",
        "Apply Link": url,
    })

seen_tc = set()
deduped = []
for r in results:
    key = (r["Title"].lower().strip(), r["Company"].lower().strip())
    if key not in seen_tc:
        seen_tc.add(key)
        deduped.append(r)

print(f"Interim pass: {len(results)} scored rows, {len(deduped)} after title/company dedup.")
print("The final export cell re-runs collection, scoring, dedup and the Excel build.")


In [ ]:
# Old duplicate Excel export cell disabled.
# This prevents Colab from downloading an older workbook without the Networking Tracker tab.
print("Skipped old duplicate Excel export. Use the final downloaded workbook from the final cell.")


In [ ]:
import requests
from bs4 import BeautifulSoup
import time

headers_req = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}

ubs_jobs = []

# UBS Workday API endpoints to try
UBS_URLS = [
    "https://jobs.ubs.com/TGNewUI/Search/home/HomeWithPreLoad?partnerid=25008&siteid=5012&PageType=JobListing&noback=1#",
    "https://jobs.ubs.com/TGNewUI/Search/home/HomeWithPreLoad?partnerid=25008&siteid=5012",
    "https://ubs.wd3.myworkdayjobs.com/UBS_Career/jobs",
    "https://ubs.wd3.myworkdayjobs.com/en-US/UBS_Career",
]

# Workday API format
WORKDAY_API_URLS = [
    "https://ubs.wd3.myworkdayjobs.com/wday/cxs/ubs/UBS_Career/jobs",
    "https://ubs.wd3.myworkdayjobs.com/wday/cxs/ubs/UBS_Career/jobs?offset=0&limit=20&searchText=data+governance+London",
]

RELEVANT = RELEVANT_TITLES  # candidate taxonomy from the settings cell

EXCLUDE = HARD_EXCLUDE      # context-aware filtering via title_is_excluded()

NON_UK_CHECK = [
    "united states", "new york", "zurich", "switzerland",
    "singapore", "hong kong", "frankfurt", "paris",
    "sydney", "tokyo", "toronto", "mumbai", "india",
]

def is_uk_ubs(loc):
    if not loc: return True
    return not any(x in loc.lower() for x in NON_UK_CHECK)

print("Checking UBS career pages...")
print("-" * 55)

# Try Workday API first
for url in WORKDAY_API_URLS:
    try:
        r = requests.post(url,
            headers={
                **headers_req,
                "Accept": "application/json",
                "Content-Type": "application/json",
            },
            json={"appliedFacets": {}, "limit": 20, "offset": 0, "searchText": "data governance London"},
            timeout=12
        )
        print(f"  Workday API: {url} â†’ {r.status_code}")
        if r.status_code == 200:
            data = r.json()
            jobs = data.get("jobPostings", [])
            print(f"    Jobs found: {len(jobs)}")
            for job in jobs:
                title = job.get("title", "")
                location = job.get("locationsText", "")
                job_url = "https://ubs.wd3.myworkdayjobs.com" + job.get("externalPath", "")
                if not any(w in title.lower() for w in RELEVANT): continue
                if title_is_excluded(title): continue
                if not is_uk_ubs(location): continue
                ubs_jobs.append({
                    "title": title, "company": "UBS",
                    "location": location, "salary": "See listing",
                    "posted": "Live now", "desc": title,
                    "source": "Career Page", "url": job_url,
                })
                print(f"    âœ… {title} â€” {location}")
    except Exception as e:
        print(f"  Workday API error: {e}")
    time.sleep(1)

# Try scraping directly
for url in UBS_URLS:
    try:
        r = requests.get(url, headers=headers_req, timeout=12)
        print(f"  Direct: {url[:60]} â†’ {r.status_code} | Size: {len(r.text)}")
        if r.status_code == 200 and len(r.text) > 1000:
            soup = BeautifulSoup(r.text, "html.parser")
            found = 0
            for a in soup.find_all("a", href=True):
                title = a.get_text(strip=True)
                href = a["href"]
                if len(title) < 8 or len(title) > 110: continue
                if not any(w in title.lower() for w in RELEVANT): continue
                if title_is_excluded(title): continue
                full_url = href if href.startswith("http") else f"https://jobs.ubs.com{href}"
                ubs_jobs.append({
                    "title": title, "company": "UBS",
                    "location": "London/UK", "salary": "See listing",
                    "posted": "Live now", "desc": title,
                    "source": "Career Page", "url": full_url,
                })
                found += 1
            print(f"    Jobs found via scrape: {found}")
    except Exception as e:
        print(f"  Direct error: {e}")
    time.sleep(1)

# Try specific London strategy search
UBS_SEARCH_URLS = [
    "https://jobs.ubs.com/TGNewUI/Search/home/HomeWithPreLoad?partnerid=25008&siteid=5012&PageType=JobListing&noback=1&SearchCriteria=strategy+operations&SearchLocation=London",
    "https://ubs.wd3.myworkdayjobs.com/en-US/UBS_Career?q=data+governance&locations=London",
]

for url in UBS_SEARCH_URLS:
    try:
        r = requests.get(url, headers=headers_req, timeout=12)
        print(f"  Search URL: {url[:60]} â†’ {r.status_code} | Size: {len(r.text)}")
        if r.status_code == 200:
            soup = BeautifulSoup(r.text, "html.parser")
            text_preview = soup.get_text()[:500]
            print(f"    Preview: {text_preview[:200]}")
    except Exception as e:
        print(f"  Search error: {e}")

print()
print("=" * 55)
print(f"UBS roles found: {len(ubs_jobs)}")
for j in ubs_jobs:
    print(f"  - {j['title']} â€” {j['location']}")

In [ ]:
import requests
import json

headers_workday = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "application/json",
    "Content-Type": "application/json",
    "X-Calypso-CSRF-Token": "undefined",
}

# Correct Workday API format
WORKDAY_COMPANIES = {
    "UBS":           "https://ubs.wd3.myworkdayjobs.com/wday/cxs/ubs/UBS_Career/jobs",
    "JP Morgan":     "https://jpmc.fa.us2.oraclecloud.com/hcmRestApi/resources/latest/recruitingCEJobRequisitions",
    "NatWest":       "https://natwest.wd3.myworkdayjobs.com/wday/cxs/natwest/NatWestGroupCareers/jobs",
    "Lloyds":        "https://lbg.wd3.myworkdayjobs.com/wday/cxs/lbg/LBG_External/jobs",
    "Santander":     "https://santander.wd3.myworkdayjobs.com/wday/cxs/santander/SantanderCareers/jobs",
    "KPMG":          "https://kpmg.wd3.myworkdayjobs.com/wday/cxs/kpmg/KPMG_UK_Careers/jobs",
    "PwC":           "https://pwc.wd3.myworkdayjobs.com/wday/cxs/pwc/Global_Campus_Experienced/jobs",
    "Mastercard":    "https://mastercard.wd1.myworkdayjobs.com/wday/cxs/mastercard/CorporateCareers/jobs",
    "Visa":          "https://visa.wd1.myworkdayjobs.com/wday/cxs/visa/Visa_Careers/jobs",
    "Alvarez Marsal":"https://alvarezandmarsal.wd1.myworkdayjobs.com/wday/cxs/alvarezandmarsal/alvarezandmarsal/jobs",
    "McKinsey":      "https://mckinsey.wd1.myworkdayjobs.com/wday/cxs/mckinsey/McKinsey_Careers/jobs",
    "Oliver Wyman":  "https://mmc.wd1.myworkdayjobs.com/wday/cxs/mmc/Oliver_Wyman_Careers/jobs",
    "Kearney":       "https://kearney.wd1.myworkdayjobs.com/wday/cxs/kearney/Kearney_Careers/jobs",
}

SEARCH_TERMS = [
    "data governance", "data quality", "regulatory reporting",
    "data management", "risk data", "management information",
    "business intelligence", "business analyst data",
]

def is_uk_role(loc):
    if not loc: return True
    return not any(x in loc.lower() for x in NON_UK)

workday_jobs = []
print("Testing Workday API for all companies...")
print("-" * 55)

for company, base_url in WORKDAY_COMPANIES.items():
    found_any = False
    for search_term in SEARCH_TERMS[:2]:  # Test first 2 terms
        payload = {
            "appliedFacets": {},
            "limit": 20,
            "offset": 0,
            "searchText": search_term,
        }
        try:
            r = requests.post(base_url, headers=headers_workday,
                            json=payload, timeout=12)
            if r.status_code == 200:
                data = r.json()
                jobs = data.get("jobPostings", [])
                found = 0
                for job in jobs:
                    title = job.get("title", "")
                    location = job.get("locationsText", "")
                    path = job.get("externalPath", "")
                    domain = "/".join(base_url.split("/")[:3])
                    job_url = domain + path
                    if not any(w in title.lower() for w in RELEVANT): continue
                    if title_is_excluded(title): continue
                    if not is_uk_role(location): continue
                    workday_jobs.append({
                        "title": title, "company": company,
                        "location": location, "salary": "See listing",
                        "posted": "Live now", "desc": title,
                        "source": "Career Page", "url": job_url,
                    })
                    found += 1
                if jobs:
                    print(f"  âœ… {company:<20} API works | Total jobs: {len(jobs)} | Relevant: {found}")
                    found_any = True
                    break
            elif r.status_code == 422:
                # Try GET instead of POST
                get_url = f"{base_url}?searchText={search_term.replace(' ', '+')}&limit=20"
                r2 = requests.get(get_url, headers=headers_workday, timeout=12)
                if r2.status_code == 200:
                    data = r2.json()
                    jobs = data.get("jobPostings", [])
                    print(f"  âœ… {company:<20} GET works | Jobs: {len(jobs)}")
                    found_any = True
                    break
                else:
                    print(f"  âŒ {company:<20} {r.status_code} POST, {r2.status_code} GET")
                    break
            else:
                print(f"  âŒ {company:<20} {r.status_code}")
                break
        except Exception as e:
            print(f"  âŒ {company:<20} Error: {str(e)[:50]}")
            break

print()
print("=" * 55)
print(f"Workday roles found: {len(workday_jobs)}")
for j in workday_jobs[:20]:
    print(f"  - {j['title']} @ {j['company']} â€” {j['location']}")

In [ ]:
import requests
import time

headers_workday = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "application/json",
    "Content-Type": "application/json",
}

WORKDAY_WORKING = {
    "Santander":  "https://santander.wd3.myworkdayjobs.com/wday/cxs/santander/SantanderCareers/jobs",
    "Mastercard": "https://mastercard.wd1.myworkdayjobs.com/wday/cxs/mastercard/CorporateCareers/jobs",
}

# Try other Workday slugs with corrected formats
WORKDAY_RETRY = {
    "UBS":        "https://ubs.wd3.myworkdayjobs.com/wday/cxs/ubs/UBS_Career/jobs",
    "NatWest":    "https://natwest.wd3.myworkdayjobs.com/wday/cxs/natwest/NatWestGroupCareers/jobs",
    "KPMG":       "https://kpmg.wd3.myworkdayjobs.com/wday/cxs/kpmg/KPMG_UK_Careers/jobs",
    "Visa":       "https://visa.wd1.myworkdayjobs.com/wday/cxs/visa/Visa_Careers/jobs",
    "McKinsey":   "https://mckinsey.wd1.myworkdayjobs.com/wday/cxs/mckinsey/McKinsey_Careers/jobs",
    "Kearney":    "https://kearney.wd1.myworkdayjobs.com/wday/cxs/kearney/Kearney_Careers/jobs",
    "Lloyds":     "https://lbg.wd3.myworkdayjobs.com/wday/cxs/lbg/LBG_External/jobs",
    "PwC":        "https://pwc.wd3.myworkdayjobs.com/wday/cxs/pwc/Global_Campus_Experienced/jobs",
    "Oliver Wyman": "https://mmc.wd1.myworkdayjobs.com/wday/cxs/mmc/Oliver_Wyman_Careers/jobs",
    "Barclays":   "https://barclays.wd3.myworkdayjobs.com/wday/cxs/barclays/External/jobs",
    "HSBC":       "https://hsbc.wd3.myworkdayjobs.com/wday/cxs/hsbc/HSBCGlobalCareers/jobs",
    "Goldman":    "https://goldmansachs.wd1.myworkdayjobs.com/wday/cxs/goldmansachs/EmployeeJobPostings/jobs",
    "JP Morgan":  "https://jpmc.wd1.myworkdayjobs.com/wday/cxs/jpmc/JPMorganChase/jobs",
    "Deloitte":   "https://deloitte.wd1.myworkdayjobs.com/wday/cxs/deloitte/DTL_External/jobs",
    "BCG":        "https://bcg.wd3.myworkdayjobs.com/wday/cxs/bcg/BCG_Career_Internal_Staff_-_0/jobs",
    "Accenture":  "https://accenture.wd3.myworkdayjobs.com/wday/cxs/accenture/AccentureCareers/jobs",
}

RELEVANT = RELEVANT_TITLES  # candidate taxonomy from the settings cell
EXCLUDE = HARD_EXCLUDE      # context-aware filtering via title_is_excluded()

UK_LOCATIONS = ["london", "uk", "united kingdom", "england",
                "remote", "hybrid", "manchester", "edinburgh",
                "birmingham", "bristol", "leeds"]

def is_uk_role(loc):
    if not loc: return True
    loc_lower = loc.lower()
    # Must contain a UK location indicator
    return any(x in loc_lower for x in UK_LOCATIONS)

workday_jobs = []

# â”€â”€ Pull from confirmed working APIs with London filter â”€â”€â”€â”€â”€â”€â”€
print("Pulling from confirmed Workday APIs...")
print("-" * 55)

SEARCH_TERMS = [
    "data governance", "data quality", "regulatory reporting",
    "data management", "risk data", "management information",
    "business intelligence", "business analyst data",
]

for company, base_url in WORKDAY_WORKING.items():
    found_total = 0
    for search_term in SEARCH_TERMS:
        payload = {
            "appliedFacets": {},
            "limit": 20,
            "offset": 0,
            "searchText": search_term,
        }
        try:
            r = requests.post(base_url, headers=headers_workday,
                            json=payload, timeout=12)
            if r.status_code == 200:
                data = r.json()
                jobs = data.get("jobPostings", [])
                for job in jobs:
                    title = job.get("title", "")
                    location = job.get("locationsText", "")
                    path = job.get("externalPath", "")
                    domain = "/".join(base_url.split("/")[:3])
                    job_url = domain + path
                    if not any(w in title.lower() for w in RELEVANT): continue
                    if title_is_excluded(title): continue
                    if not is_uk_role(location): continue
                    # Avoid duplicates
                    if any(j["url"] == job_url for j in workday_jobs): continue
                    workday_jobs.append({
                        "title": title, "company": company,
                        "location": location, "salary": "See listing",
                        "posted": "Live now", "desc": title,
                        "source": "Career Page", "url": job_url,
                    })
                    found_total += 1
        except Exception as e:
            pass
        time.sleep(0.5)
    print(f"  âœ… {company:<20} {found_total} UK relevant roles")

# â”€â”€ Retry failed companies with corrected slugs â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print()
print("Retrying failed companies with corrected Workday slugs...")
print("-" * 55)

for company, base_url in WORKDAY_RETRY.items():
    try:
        payload = {
            "appliedFacets": {},
            "limit": 20,
            "offset": 0,
            "searchText": "data governance London",
        }
        r = requests.post(base_url, headers=headers_workday,
                         json=payload, timeout=12)
        if r.status_code == 200:
            data = r.json()
            jobs = data.get("jobPostings", [])
            found = 0
            for job in jobs:
                title = job.get("title", "")
                location = job.get("locationsText", "")
                path = job.get("externalPath", "")
                domain = "/".join(base_url.split("/")[:3])
                job_url = domain + path
                if not any(w in title.lower() for w in RELEVANT): continue
                if title_is_excluded(title): continue
                if not is_uk_role(location): continue
                if any(j["url"] == job_url for j in workday_jobs): continue
                workday_jobs.append({
                    "title": title, "company": company,
                    "location": location, "salary": "See listing",
                    "posted": "Live now", "desc": title,
                    "source": "Career Page", "url": job_url,
                })
                found += 1
            print(f"  âœ… {company:<20} {r.status_code} | {len(jobs)} total | {found} UK relevant")
        else:
            print(f"  âŒ {company:<20} {r.status_code}")
    except Exception as e:
        print(f"  âŒ {company:<20} Error: {str(e)[:40]}")
    time.sleep(0.5)

print()
print("=" * 55)
print(f"Total Workday UK roles: {len(workday_jobs)}")
print()
print("Roles found:")
for j in workday_jobs:
    print(f"  [{j['company']}] {j['title']} â€” {j['location']}")

In [ ]:
import requests
import time

# Fix 422 errors â€” Workday requires CSRF token
# Step 1: Get CSRF token from the page first, then use it in API call

headers_browser = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-GB,en;q=0.9",
}

CSRF_COMPANIES = {
    "UBS":       ("https://ubs.wd3.myworkdayjobs.com/en-US/UBS_Career",
                  "https://ubs.wd3.myworkdayjobs.com/wday/cxs/ubs/UBS_Career/jobs"),
    "NatWest":   ("https://natwest.wd3.myworkdayjobs.com/en-GB/NatWestGroupCareers",
                  "https://natwest.wd3.myworkdayjobs.com/wday/cxs/natwest/NatWestGroupCareers/jobs"),
    "KPMG":      ("https://kpmg.wd3.myworkdayjobs.com/en-GB/KPMG_UK_Careers",
                  "https://kpmg.wd3.myworkdayjobs.com/wday/cxs/kpmg/KPMG_UK_Careers/jobs"),
    "HSBC":      ("https://hsbc.wd3.myworkdayjobs.com/en-GB/HSBCGlobalCareers",
                  "https://hsbc.wd3.myworkdayjobs.com/wday/cxs/hsbc/HSBCGlobalCareers/jobs"),
    "Goldman":   ("https://goldmansachs.wd1.myworkdayjobs.com/en-US/EmployeeJobPostings",
                  "https://goldmansachs.wd1.myworkdayjobs.com/wday/cxs/goldmansachs/EmployeeJobPostings/jobs"),
    "JP Morgan": ("https://jpmc.wd1.myworkdayjobs.com/en-US/JPMorganChase",
                  "https://jpmc.wd1.myworkdayjobs.com/wday/cxs/jpmc/JPMorganChase/jobs"),
    "Deloitte":  ("https://deloitte.wd1.myworkdayjobs.com/en-US/DTL_External",
                  "https://deloitte.wd1.myworkdayjobs.com/wday/cxs/deloitte/DTL_External/jobs"),
    "BCG":       ("https://bcg.wd3.myworkdayjobs.com/en-US/BCG_Career_Internal_Staff_-_0",
                  "https://bcg.wd3.myworkdayjobs.com/wday/cxs/bcg/BCG_Career_Internal_Staff_-_0/jobs"),
    "McKinsey":  ("https://mckinsey.wd1.myworkdayjobs.com/en-US/McKinsey_Careers",
                  "https://mckinsey.wd1.myworkdayjobs.com/wday/cxs/mckinsey/McKinsey_Careers/jobs"),
    "Accenture": ("https://accenture.wd3.myworkdayjobs.com/en-US/AccentureCareers",
                  "https://accenture.wd3.myworkdayjobs.com/wday/cxs/accenture/AccentureCareers/jobs"),
    "Visa":      ("https://visa.wd1.myworkdayjobs.com/en-US/Visa_Careers",
                  "https://visa.wd1.myworkdayjobs.com/wday/cxs/visa/Visa_Careers/jobs"),
    "Kearney":   ("https://kearney.wd1.myworkdayjobs.com/en-US/Kearney_Careers",
                  "https://kearney.wd1.myworkdayjobs.com/wday/cxs/kearney/Kearney_Careers/jobs"),
}

RELEVANT = RELEVANT_TITLES  # candidate taxonomy from the settings cell
EXCLUDE = HARD_EXCLUDE      # context-aware filtering via title_is_excluded()
UK_LOCATIONS = [
    "london", "uk", "united kingdom", "england",
    "remote", "hybrid", "manchester", "edinburgh",
    "birmingham", "bristol", "leeds", "sheffield",
]

def is_uk_role(loc):
    if not loc: return True
    return any(x in loc.lower() for x in UK_LOCATIONS)

csrf_jobs = []
print("Getting CSRF tokens and pulling Workday APIs...")
print("-" * 55)

session = requests.Session()

for company, (page_url, api_url) in CSRF_COMPANIES.items():
    try:
        # Step 1: Visit the careers page to get cookies + CSRF token
        page_r = session.get(page_url, headers=headers_browser, timeout=12)

        # Extract CSRF token from cookies or response
        csrf_token = None
        for cookie in session.cookies:
            if "csrf" in cookie.name.lower() or "token" in cookie.name.lower():
                csrf_token = cookie.value
                break

        # Also check response headers
        if not csrf_token:
            csrf_token = page_r.headers.get("X-Calypso-CSRF-Token", "undefined")

        # Step 2: Call API with CSRF token
        api_headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
            "Accept": "application/json",
            "Content-Type": "application/json",
            "X-Calypso-CSRF-Token": csrf_token or "undefined",
            "Referer": page_url,
            "Origin": "/".join(page_url.split("/")[:3]),
        }

        for search_term in ["data governance London", "regulatory reporting London", "data quality London"]:
            payload = {
                "appliedFacets": {},
                "limit": 20,
                "offset": 0,
                "searchText": search_term,
            }
            api_r = session.post(api_url, headers=api_headers,
                                json=payload, timeout=12)

            if api_r.status_code == 200:
                data = api_r.json()
                jobs = data.get("jobPostings", [])
                found = 0
                for job in jobs:
                    title = job.get("title", "")
                    location = job.get("locationsText", "")
                    path = job.get("externalPath", "")
                    domain = "/".join(api_url.split("/")[:3])
                    job_url = domain + path
                    if not any(w in title.lower() for w in RELEVANT): continue
                    if title_is_excluded(title): continue
                    if not is_uk_role(location): continue
                    if any(j["url"] == job_url for j in csrf_jobs): continue
                    csrf_jobs.append({
                        "title": title, "company": company,
                        "location": location, "salary": "See listing",
                        "posted": "Live now", "desc": title,
                        "source": "Career Page", "url": job_url,
                    })
                    found += 1
                if jobs:
                    print(f"  âœ… {company:<15} {api_r.status_code} | {len(jobs)} total | {found} UK relevant | CSRF: {csrf_token[:10] if csrf_token else 'none'}")
                    break
            else:
                print(f"  âŒ {company:<15} {api_r.status_code} | CSRF: {csrf_token[:10] if csrf_token else 'none'}")
                break
            time.sleep(0.5)

    except Exception as e:
        print(f"  âŒ {company:<15} Error: {str(e)[:50]}")
    time.sleep(1)

# Add all Workday jobs to company_jobs
all_workday = workday_jobs + csrf_jobs
company_jobs += all_workday

print()
print("=" * 55)
print(f"New roles from CSRF fix:     {len(csrf_jobs)}")
print(f"Total Workday roles:         {len(all_workday)}")
print(f"Total company jobs now:      {len(company_jobs)}")
print()
if csrf_jobs:
    print("New roles found:")
    for j in csrf_jobs:
        print(f"  [{j['company']}] {j['title']} â€” {j['location']}")

In [ ]:
# Add company-specific LinkedIn searches for blocked Workday companies
TARGETED_LI = [
    "Barclays+data+governance+London",
    "HSBC+regulatory+reporting+London",
    "NatWest+data+quality+London",
    "Lloyds+data+management+London",
    "Santander+regulatory+data+London",
    "Standard+Chartered+data+governance+London",
    "Deloitte+data+governance+analyst+London",
    "KPMG+regulatory+reporting+analyst+London",
    "EY+data+governance+London",
    "PwC+risk+data+analyst+London",
    "LSEG+reference+data+analyst+London",
    "Experian+data+quality+London",
    "Revolut+data+quality+London",
    "Monzo+data+governance+London",
    "Quantexa+data+analyst+London",
]

HEADERS_LI = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

print("Pulling company-specific LinkedIn searches...")
print("-" * 55)

extra_li = []
seen_li_extra = set()

for kw in TARGETED_LI:
    url = (f"https://www.linkedin.com/jobs/search/?keywords={kw}"
           f"&location=London%2C+United+Kingdom&f_TPR=r604800")
    try:
        r = requests.get(url, headers=HEADERS_LI, timeout=15)
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.find_all("div", class_=lambda x: x and "base-card" in str(x))
        found = 0
        for card in cards:
            title_tag = card.find("h3", class_=lambda x: x and "base-search-card__title" in str(x))
            title = title_tag.get_text(strip=True) if title_tag else ""
            company_tag = card.find("h4", class_=lambda x: x and "base-search-card__subtitle" in str(x))
            company = company_tag.get_text(strip=True) if company_tag else ""
            loc_tag = card.find("span", class_=lambda x: x and "job-search-card__location" in str(x))
            location = loc_tag.get_text(strip=True) if loc_tag else "London"
            link_tag = card.find("a", class_=lambda x: x and "base-card__full-link" in str(x))
            link = link_tag["href"].split("?")[0] if link_tag else ""
            if not title or not company or not link: continue
            if link in seen_li_extra: continue
            # High recall: do not require title keyword match before AI review
            if title_is_excluded(title): continue
            seen_li_extra.add(link)
            extra_li.append({
                "title": title, "company": company,
                "location": location, "salary": "See listing",
                "posted": "This week", "desc": title,
                "source": "LinkedIn", "url": link,
            })
            found += 1
        print(f"  {kw.replace('+', ' ')[:45]:<45} {found} roles")
        time.sleep(2)
    except Exception as e:
        print(f"  {kw[:45]} blocked")

# Add to linkedin_jobs
linkedin_jobs += extra_li

print("-" * 55)
print(f"Extra LinkedIn roles: {len(extra_li)}")
print(f"Total LinkedIn roles: {len(linkedin_jobs)}")


In [ ]:
import pandas as pd
from google.colab import files
from collections import Counter
import os
import json
import re
import requests

results = []
seen_urls = set()

STRICT_NON_UK_TERMS = [
    "united states", " usa", " us,", "u.s.", "canada", "australia", "japan",
    "singapore", "india", "germany", "france", "spain", "italy", "mexico", "brazil",
    "new york", "san francisco", "seattle", "boston", "los angeles", "washington",
    "chicago", "dallas", "miami", "austin", "tokyo", "sydney", "melbourne",
    "toronto", "paris", "berlin", "barcelona", "rome", "gurugram", "gurgaon",
    "mumbai", "bangalore", "bengaluru", "munich", "muenchen", "m?nchen",
    "frankfurt", "dublin", "ireland", "portugal", "lisbon", "sweden", "stockholm",
    "netherlands", "belgium", "poland", "hyderabad", "delhi", "sofia", "riga",
    "warsaw", "amsterdam", "madrid", "milan", "shanghai", "beijing", "hong kong",
    "seoul", "dubai", "abu dhabi", "riyadh", "apac", "emea", "latam",
    "remote - us", "remote us",
]

def strict_is_uk_loc(loc):
    loc_lower = normalise_text(loc)
    if not loc_lower:
        return True
    if any(term in loc_lower for term in STRICT_NON_UK_TERMS):
        return False
    country, _ = get_country_continent(loc_lower)
    if country in NON_UK_COUNTRIES:
        return False
    return True

is_uk_loc = strict_is_uk_loc


# ── UK Visa Sponsor Register ──────────────────────────────────────────────────
# Downloads the Home Office register of licensed Skilled Worker sponsors once
# per session and caches it in _uk_sponsor_cache. Falls back gracefully if the
# download fails so the rest of the pipeline is never blocked.
_uk_sponsor_cache = None

def load_uk_sponsor_register():
    """
    Downloads the Home Office register of licensed Skilled Worker sponsors.
    The asset URL changes with each publication, so we scrape the landing page
    for the current XLSX link rather than hardcoding it.
    """
    global _uk_sponsor_cache
    if _uk_sponsor_cache is not None:
        return _uk_sponsor_cache

    # The gov.uk page renders attachment links via JavaScript, so BeautifulSoup can't
    # find them. Use the GOV.UK Content API (plain JSON, no JS) instead.
    CONTENT_API = (
        "https://www.gov.uk/api/content/government/publications/"
        "register-of-licensed-sponsors-workers"
    )
    try:
        import io
        api_resp = requests.get(CONTENT_API, timeout=20, headers={"User-Agent": "Mozilla/5.0"})
        api_resp.raise_for_status()
        content = api_resp.json()

        xlsx_link = None
        # details.documents is a list of HTML strings on gov.uk, not dicts
        import re as _re
        for doc in content.get("details", {}).get("documents", []):
            if isinstance(doc, dict):
                url_candidate = doc.get("url", "")
                if url_candidate.endswith(".xlsx"):
                    xlsx_link = url_candidate
                    break
            elif isinstance(doc, str):
                m = _re.search(r'https://assets\.publishing\.service\.gov\.uk[^"\'>\s]+\.xlsx', doc)
                if m:
                    xlsx_link = m.group(0)
                    break
        # Final fallback: scan the entire raw JSON response
        if not xlsx_link:
            matches = _re.findall(r'https://assets\.publishing\.service\.gov\.uk[^"\'>\s]+\.xlsx', api_resp.text)
            if matches:
                xlsx_link = matches[0]

        if not xlsx_link:
            print("  ⚠️  UK sponsor register: could not locate XLSX via GOV.UK Content API.")
            _uk_sponsor_cache = set()
            return _uk_sponsor_cache

        r = requests.get(xlsx_link, timeout=90, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        df = pd.read_excel(io.BytesIO(r.content), header=0)
        name_col = next(
            (c for c in df.columns if "organisation" in c.lower() or "name" in c.lower()),
            None,
        )
        if name_col:
            _uk_sponsor_cache = set(df[name_col].dropna().str.lower().str.strip())
            print(f"  ✅ UK sponsor register loaded: {len(_uk_sponsor_cache):,} companies")
        else:
            print(f"  ⚠️  UK sponsor register: unexpected columns {list(df.columns)[:5]}. Falling back.")
            _uk_sponsor_cache = set()
    except Exception as e:
        print(f"  ⚠️  UK sponsor register download failed: {e}. Text-only fallback active.")
        _uk_sponsor_cache = set()
    return _uk_sponsor_cache


def sponsorship_risk_enhanced(text, company=""):
    """
    Priority order:
      1. Explicit phrases in the job description.
      2. Cross-reference against the UK Home Office licensed sponsor register.
    """
    t = normalise_text(text)
    no_sponsorship_phrases = [
        "no sponsorship", "cannot sponsor", "unable to sponsor", "will not sponsor",
        "does not sponsor", "do not sponsor", "must have right to work",
        "right to work in the uk", "eligible to work in the uk", "existing right to work",
        "without sponsorship", "sponsorship is not available",
    ]
    positive_phrases = [
        "visa sponsorship", "sponsorship available", "skilled worker", "sponsor licence",
        "graduate visa",
    ]
    if any(phrase in t for phrase in no_sponsorship_phrases):
        return "High - right to work/no sponsorship wording"
    if any(phrase in t for phrase in positive_phrases):
        return "Low - sponsorship/visa friendly signal"

    register = load_uk_sponsor_register()
    if register and company:
        co_clean = normalise_text(company).replace(" (agency)", "").strip()
        if co_clean in register:
            return "Low - on UK sponsor register"
        co_tokens = [tok for tok in co_clean.split() if len(tok) > 3]
        if co_tokens and any(all(tok in entry for tok in co_tokens) for entry in register):
            return "Low - on UK sponsor register (partial match)"

    return "Unknown"


# ── CANDIDATE CAPABILITY MODEL ───────────────────────────────────────────────
# Dimensions map to what the CV evidences: data governance, data quality,
# regulatory reporting, reconciliation, MI/BI, business analysis, stakeholder
# management and project documentation.
CAPABILITY_DIMENSIONS = {
    "Data Governance": ["data governance", "information governance", "data steward", "data owner",
                        "data policy", "governance framework", "data catalogue", "data catalog"],
    "Data Quality": ["data quality", "data validation", "data integrity", "data remediation",
                     "data profiling", "data cleansing", "root cause"],
    "Data Management & MDM": ["data management", "master data", "mdm", "reference data",
                              "metadata", "data lineage", "data dictionary"],
    "Regulatory Reporting": ["regulatory reporting", "regulatory return", "regulatory submission",
                             "corep", "finrep", "fr y-14", "fry-14", "ccar", "prudential"],
    "Risk & Regulatory Data": ["risk data", "regulatory data", "bcbs", "basel", "risk reporting",
                               "credit risk", "risk framework", "fca", "pra "],
    "Reconciliation & Controls": ["reconciliation", "reconcile", "month-end", "month end",
                                  "controls", "control framework", "variance", "break resolution"],
    "Financial Services Domain": ["bank", "banking", "financial services", "capital markets",
                                  "asset management", "insurance", "payments", "fintech", "lending"],
    "SQL & Data Querying": ["sql", "queries", "query", "database", "data extraction", "joins"],
    "BI & Visualisation": ["power bi", "powerbi", "tableau", "dashboard", "visualisation",
                           "visualization", "qlik", "looker"],
    "Python & Automation": ["python", "automation", "vba", "macros", "alteryx", "scripting"],
    "Business Analysis": ["business analysis", "business analyst", "requirements", "process mapping",
                          "process improvement", "uat", "gap analysis"],
    "MI & Reporting": ["management information", "mi reporting", "reporting pack", "reporting cycle",
                       "ad-hoc reporting", "ad hoc reporting", "insight"],
    "Stakeholder Management": ["stakeholder", "cross-functional", "senior leaders", "executive",
                               "business partner", "communication", "presentation"],
    "Project & PMO": ["project", "programme", "program", "pmo", "milestone", "raid", "jira",
                      "agile", "waterfall", "prince2"],
    "Documentation & Audit": ["documentation", "audit trail", "audit-ready", "evidence",
                              "sop", "procedures", "record keeping", "sharepoint", "servicenow"],
}


def score_capability_dimensions(job_row):
    text = normalise_text(" ".join([
        job_row.get("Title", ""), job_row.get("Company", ""),
        job_row.get("Job Description", ""), job_row.get("Why It Matches", ""),
    ]))
    scores = {}
    for dimension, keywords in CAPABILITY_DIMENSIONS.items():
        hits = sum(1 for keyword in keywords if keyword in text)
        # Two distinct signals in a dimension is already a strong indication.
        scores[dimension] = min(10, round((hits / 2) * 10, 1))
    return scores


# How central each capability is to this candidate's career, used to turn the
# dimension scores into a single "how close is this to the middle of her CV"
# rating. Financial Services Domain is applied separately as a modifier.
CAREER_ALIGNMENT_VALUE = {
    "Data Governance": 10,
    "Data Quality": 10,
    "Regulatory Reporting": 10,
    "Data Management & MDM": 9,
    "Risk & Regulatory Data": 9,
    "Reconciliation & Controls": 8,
    "MI & Reporting": 8,
    "SQL & Data Querying": 7,
    "Business Analysis": 7,
    "BI & Visualisation": 7,
    "Documentation & Audit": 6,
    "Python & Automation": 5,
    "Project & PMO": 5,
    "Stakeholder Management": 4,
}


def capability_fit_score(dimensions):
    """
    Depth, not breadth. Averaging all fifteen dimensions punished a focused
    advert that matches the CV exactly, so only the strongest five count.
    """
    top = sorted(dimensions.values(), reverse=True)[:5]
    return round(sum(top) / max(1, len(top)), 1)


def career_alignment_score(dimensions):
    """
    Driven by the best-evidenced capabilities weighted by how central each one is
    to this CV, plus a financial-services modifier. A pure regulatory reporting
    role scores high even though it touches only one dimension.
    """
    ranked = sorted(
        (CAREER_ALIGNMENT_VALUE[dimension] * (score / 10.0)
         for dimension, score in dimensions.items() if dimension in CAREER_ALIGNMENT_VALUE),
        reverse=True,
    )
    best = ranked[0] if ranked else 0
    second = ranked[1] if len(ranked) > 1 else 0
    domain_modifier = 0.1 * dimensions.get("Financial Services Domain", 0)
    return min(10, round(0.75 * best + 0.25 * second + domain_modifier, 1))


def classify_job_family(job_row):
    text = normalise_text(" ".join([
        job_row.get("Title", ""), job_row.get("Job Description", ""), job_row.get("Why It Matches", ""),
    ]))
    families = {
        "Data Governance": ["data governance", "data steward", "metadata", "data lineage", "data policy"],
        "Data Quality": ["data quality", "data validation", "data remediation", "data integrity"],
        "Data Management": ["data management", "master data", "mdm", "reference data"],
        "Regulatory Reporting": ["regulatory reporting", "corep", "finrep", "fr y-14", "prudential", "regulatory return"],
        "Risk Data": ["risk data", "risk reporting", "credit risk", "bcbs", "basel"],
        "Financial Data & Reconciliation": ["financial data", "reconciliation", "month-end", "financial reporting"],
        "Business Analysis": ["business analyst", "business analysis", "requirements", "process analyst"],
        "MI & BI Reporting": ["management information", "mi analyst", "business intelligence", "power bi", "tableau", "dashboard"],
        "Analytics": ["analytics", "insight", "data analysis"],
        "Project / PMO": ["pmo", "project analyst", "programme", "portfolio", "project support"],
        "Change & Transformation": ["transformation", "change analyst", "business change", "target operating model"],
        "Consulting": ["consultant", "consulting", "advisory", "client delivery"],
    }
    tags = [family for family, keywords in families.items() if any(keyword in text for keyword in keywords)]
    return ", ".join(tags[:4]) if tags else "General Analyst"


def classify_company_type(company):
    c = normalise_text(company)
    if any(x in c for x in ["barclays", "natwest", "hsbc", "lloyds", "santander", "ubs", "jp morgan",
                            "jpmorgan", "goldman", "citi", "nationwide", "metro bank", "starling",
                            "monzo bank", "standard chartered", "bank of"]):
        return "Bank"
    if any(x in c for x in ["monzo", "wise", "revolut", "stripe", "adyen", "zopa", "tide", "plaid",
                            "sumup", "iwoca", "checkout", "klarna", "gocardless", "truelayer",
                            "moneybox", "oaknorth", "allica", "freetrade", "marshmallow"]):
        return "Fintech"
    if any(x in c for x in ["mckinsey", "bcg", "bain", "oliver wyman", "kearney", "deloitte", "pwc",
                            "kpmg", "ey", "accenture", "slalom", "capgemini", "alixpartners",
                            "grant thornton", "bdo", "north highland", "pa consulting"]):
        return "Consulting"
    if any(x in c for x in ["experian", "equifax", "transunion", "moody", "s&p", "lseg", "refinitiv",
                            "bloomberg", "fitch", "ice ", "morningstar", "dun & bradstreet"]):
        return "Data & Market Infrastructure"
    if any(x in c for x in ["complyadvantage", "quantexa", "onfido", "featurespace", "regtech",
                            "napier", "fenergo"]):
        return "RegTech"
    if any(x in c for x in ["aviva", "legal & general", "prudential", "axa", "allianz", "zurich insurance",
                            "hiscox", "beazley", "insurance"]):
        return "Insurance"
    if any(x in c for x in ["blackrock", "schroders", "fidelity", "m&g", "abrdn", "invesco",
                            "janus", "asset management", "investments"]):
        return "Asset Management"
    if any(x in c for x in ["google", "amazon", "microsoft", "meta", "salesforce", "oracle", "sap"]):
        return "Technology"
    if any(x in c for x in ["fca", "bank of england", "nhs", "council", "government", "ministry",
                            "hmrc", "ofgem", "public"]):
        return "Regulator / Public Sector"
    if any(x in c for x in ["university", "college", "school"]):
        return "University"
    return "Scale-up" if any(x in c for x in ["limited", "ltd", "group"]) else "Other"


def recommend_resume(job_family, dimensions, track_text=""):
    """Which CV version to lead with, and how much rework it needs."""
    family_text = normalise_text(job_family)
    if "data governance" in family_text or dimensions.get("Data Governance", 0) >= 5:
        resume = "Data Governance Resume"
    elif "data quality" in family_text or dimensions.get("Data Quality", 0) >= 5:
        resume = "Data Quality Resume"
    elif "regulatory" in family_text or dimensions.get("Regulatory Reporting", 0) >= 5:
        resume = "Regulatory Reporting Resume"
    elif "risk data" in family_text or dimensions.get("Risk & Regulatory Data", 0) >= 5:
        resume = "Risk Data Resume"
    elif "mi & bi" in family_text or dimensions.get("BI & Visualisation", 0) >= 5:
        resume = "MI / BI Reporting Resume"
    elif "business analysis" in family_text:
        resume = "Business Analysis Resume"
    elif "pmo" in family_text or "project" in family_text:
        resume = "Project / PMO Resume"
    else:
        resume = "Core Data Analyst Resume"

    changes = []
    if dimensions.get("Regulatory Reporting", 0) >= 5:
        changes.append("lead with FR Y-14 H1/H2 and regulatory submission bullets")
    if dimensions.get("Data Governance", 0) >= 5:
        changes.append("surface governance framework, metadata and lineage work")
    if dimensions.get("Data Quality", 0) >= 5:
        changes.append("foreground DQ packages and BCBS 239 validation checks")
    if dimensions.get("BI & Visualisation", 0) >= 5:
        changes.append("add Power BI / Tableau reporting examples")
    if dimensions.get("Python & Automation", 0) >= 5:
        changes.append("add Python / Alteryx automation detail")
    if dimensions.get("Project & PMO", 0) >= 5:
        changes.append("bring MSc Programme & Project Management forward")
    if not changes:
        changes.append("minor headline and skills alignment only")

    # Effort follows how far the role sits from the CV's own track, not how many
    # strengths the advert mentions - otherwise the best matches look expensive.
    track = normalise_text(track_text)
    if any(flag in track for flag in ("generic title", "not data ba", "no mapped career track")):
        effort = "High"
    elif track.startswith("primary"):
        effort = "Low"
    elif track.startswith("secondary"):
        effort = "Medium"
    else:
        effort = "High"
    return resume, effort, "; ".join(changes[:4])


def networking_recommendation(job_row, company_type):
    """Suggested warm route in. Based on the candidate's own history and study."""
    company = normalise_text(job_row.get("Company", ""))
    if "wells fargo" in company:
        return "Former employer - reach out to ex-colleagues in the data management team first."
    if "mu sigma" in company:
        return "Former employer - use the Mu Sigma alumni network for an internal referral."
    if company_type == "Bank":
        return "Find a data governance, data quality or regulatory reporting contact in the bank; Wells Fargo alumni now in UK banking are the warmest route."
    if company_type == "Fintech":
        return "Message someone in data/analytics or risk & compliance ops about how governance sits in the team."
    if company_type == "Consulting":
        return "Referral route preferred; ask about the data governance / risk & regulatory practice and the analyst entry point."
    if company_type == "RegTech":
        return "Contact a delivery or customer-facing data lead; regulatory reporting experience is the hook."
    if company_type == "Data & Market Infrastructure":
        return "Approach a data operations or data quality lead; reference data and lineage experience is directly relevant."
    if company_type == "Regulator / Public Sector":
        return "Check the published recruitment process first, then find a data governance contact for context."
    if company_type == "University":
        return "Warwick network - speak to the careers team or a course contact before applying."
    return "Find one hiring-manager-adjacent contact and ask what the reporting and data quality pain points are."


def append_rule_decision_layer(shortlisted_jobs):
    enriched = []
    for row in shortlisted_jobs:
        dimensions = score_capability_dimensions(row)
        job_family = classify_job_family(row)
        company_type = classify_company_type(row.get("Company", ""))
        resume, resume_effort, resume_delta = recommend_resume(job_family, dimensions, row.get("Why It Matches", ""))
        networking = networking_recommendation(row, company_type)

        rule_score = float(row.get("Score /100", 0) or 0)
        stretch = float(row.get("Stretch (1-10)", 10) or 10)
        capability_fit = capability_fit_score(dimensions)
        career_alignment = career_alignment_score(dimensions)

        sponsorship_penalty = 8 if "high" in normalise_text(row.get("Sponsorship risk", "")) else 0
        source_quality_bonus = {"Adzuna": 4, "Reed": 4, "Career Page": 3, "LinkedIn": 0}.get(row.get("Source", ""), 1)
        description_bonus = 4 if row.get("Description Available") == "Yes" else 0
        resume_effort_penalty = {"Low": 0, "Medium": 5, "High": 12}.get(resume_effort, 5)
        stretch_penalty = max(0, stretch - 5) * 3

        # Rows the source only gave a title for have no text for the capability
        # model to read. Lean on the rule score rather than punishing the row for
        # information the job board withheld.
        has_description = (
            row.get("Description Available") == "Yes"
            and not str(row.get("Description Quality", "")).startswith("Title Only")
        )
        rule_weight, capability_weight = (0.45, 0.20) if has_description else (0.85, 0.0)

        priority_score = round(
            rule_score * rule_weight +
            capability_fit * 10 * capability_weight +
            career_alignment * 10 * capability_weight +
            source_quality_bonus +
            description_bonus -
            resume_effort_penalty -
            stretch_penalty -
            sponsorship_penalty,
            1,
        )
        priority_score = max(0, min(100, priority_score))

        if priority_score >= 72:
            apply_decision = "YES"
            final_bucket = "A - Apply Now"
        elif priority_score >= 55:
            apply_decision = "MAYBE"
            final_bucket = "B - High Upside"
        elif priority_score >= 40:
            apply_decision = "NETWORK"
            final_bucket = "C - Network First"
        else:
            apply_decision = "NO"
            final_bucket = "D - Low Priority"

        row.update({
            "Job Family": job_family,
            "Company Type": company_type,
            "Capability Fit": capability_fit,
            "Career Alignment": career_alignment,
            "Interview Probability Final": round(priority_score, 1),
            "Resume to Use": resume,
            "Resume Changes Needed": resume_delta,
            "Resume Effort Required": resume_effort,
            "Cover Letter Effort Required": "Low" if apply_decision == "YES" and resume_effort == "Low" else "Medium",
            "Networking Recommendation": networking,
            "Priority Score": priority_score,
            "Overall Priority": 0,
            "Apply?": apply_decision,
            "Final Bucket": final_bucket,
            "Decision Reasoning": f"Rule score {rule_score}/100; capability fit {capability_fit}/10; career alignment {career_alignment}/10; stretch {stretch}/10; description {row.get('Description Quality', 'Unknown')}" + ("" if has_description else " (title only - ranked on the rule score)") + f". {row.get('Why It Matches', '')}",
            "Capability Scores": "; ".join([f"{k}: {v}" for k, v in dimensions.items() if v >= 4]) or "No strong capability signals found in available text",
        })
        enriched.append(row)

    enriched.sort(key=lambda x: -x.get("Priority Score", 0))
    for index, row in enumerate(enriched, 1):
        row["Overall Priority"] = index
    return enriched


# ── FULL JOB DESCRIPTION HANDLING ────────────────────────────────────────────
# The Excel "Job Description" column holds the COMPLETE description returned by
# the source, after HTML stripping only. It is never word-trimmed, sentence-
# trimmed or replaced by an AI summary. The only length handling here exists
# because Excel itself refuses more than 32,767 characters in one cell, so the
# overflow spills into a continuation column instead of being thrown away.
EXCEL_CELL_LIMIT = 32000
_ILLEGAL_XLSX_CHARS = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")


def clean_for_excel(value):
    """Removes control characters openpyxl cannot write. No content is trimmed."""
    if value is None:
        return ""
    return _ILLEGAL_XLSX_CHARS.sub(" ", str(value))


def split_description_for_excel(text):
    """Returns (part_1, part_2). Both parts together are the full description."""
    text = clean_for_excel(text)
    if len(text) <= EXCEL_CELL_LIMIT:
        return text, ""
    remainder = text[EXCEL_CELL_LIMIT:]
    if len(remainder) > EXCEL_CELL_LIMIT:
        remainder = remainder[:EXCEL_CELL_LIMIT - 60] + " [...continues - open the Apply Link for the rest]"
    return text[:EXCEL_CELL_LIMIT], remainder


def describe_description_quality(desc, title, source, supplied_quality=""):
    """Honest labelling of what the source actually gave us."""
    text = (desc or "").strip()
    if supplied_quality:
        base = supplied_quality
    elif not text or text.lower() == (title or "").strip().lower():
        base = "Title Only"
    elif source in ("Adzuna", "Reed"):
        base = "Board Summary"
    elif source == "LinkedIn":
        base = "Title Only"
    else:
        base = "Full API"
    if text.endswith(("…", "...")) or text.endswith("… "):
        base += " (truncated by source)"
    return base


# ── EXCEL FORMATTING: CLICKABLE LINKS + USABILITY ────────────────────────────
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font
from openpyxl.utils import get_column_letter

HYPERLINK_COLUMNS = {"Apply Link", "LinkedIn URL", "URL"}
WRAP_COLUMNS = {
    "Job Description", "Job Description (continued)", "Why It Matches",
    "Decision Reasoning", "Capability Scores", "Networking Recommendation",
    "Resume Changes Needed", "Notes",
}


def format_workbook(path):
    """
    Freezes headers, enables filters, wraps long text and turns every URL column
    into a real clickable Excel hyperlink. Cells without a usable URL are left
    blank - no placeholder links are invented.
    """
    workbook = load_workbook(path)
    link_font = Font(color="0563C1", underline="single")
    header_font = Font(bold=True)
    links_made = 0

    for worksheet in workbook.worksheets:
        if worksheet.max_row < 1:
            continue
        headers = [cell.value for cell in worksheet[1]]
        for cell in worksheet[1]:
            cell.font = header_font
            cell.alignment = Alignment(vertical="center", wrap_text=False)

        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = f"A1:{get_column_letter(worksheet.max_column)}{worksheet.max_row}"

        has_wrapped_column = False
        for index, name in enumerate(headers, start=1):
            letter = get_column_letter(index)
            if name in HYPERLINK_COLUMNS:
                worksheet.column_dimensions[letter].width = 48
                for row in range(2, worksheet.max_row + 1):
                    cell = worksheet.cell(row=row, column=index)
                    url = str(cell.value or "").strip()
                    if url.lower().startswith(("http://", "https://")):
                        cell.hyperlink = url
                        cell.font = link_font
                        cell.alignment = Alignment(vertical="top")
                        links_made += 1
                    else:
                        cell.value = None
            elif name in WRAP_COLUMNS:
                has_wrapped_column = True
                worksheet.column_dimensions[letter].width = 80
                for row in range(2, worksheet.max_row + 1):
                    worksheet.cell(row=row, column=index).alignment = Alignment(wrap_text=True, vertical="top")
            else:
                worksheet.column_dimensions[letter].width = max(12, min(30, len(str(name or "")) + 6))

        # Keep rows a readable fixed height so wrapped descriptions do not
        # stretch a row over a whole screen. Expand a row to read it in full.
        if has_wrapped_column:
            for row in range(2, worksheet.max_row + 1):
                worksheet.row_dimensions[row].height = 42

    workbook.save(path)
    return links_made


AGENCIES = [
    "reed", "hays", "robert half", "michael page", "recruitment", "talent", "staffing",
    "manpower", "adecco", "randstad", "3search", "tiger recruitment", "cedar",
    "run-time", "sphere", "carousel", "focus search", "brook street", "hexagon",
    "wild berry", "edge careers", "empro", "harnham", "eames", "morgan mckinley",
    "nigel frank", "oliver james",
]

# Company + title pairs that are a known mismatch for this CV: senior markets and
# front-office roles at the large banks, and engineering-heavy data roles at the
# data vendors.
MISMATCH_RULES = [
    {"companies": ["jp morgan", "jpmorgan", "goldman sachs", "morgan stanley", "deutsche bank", "ubs",
                   "citi", "bank of america", "bnp paribas", "lazard", "rothschild", "evercore"],
     "titles": ["vice president", " vp ", "managing director", "executive director", "quantitative",
                "structuring", "trading", "sales trade", "investment banking"]},
    {"companies": ["experian", "moody", "s&p", "lseg", "refinitiv", "bloomberg"],
     "titles": ["software", "platform engineer", "site reliability", "principal engineer"]},
    {"companies": ["skanska", "bovis", "mace", "wsp", "aecom", "balfour", "costain", "kier"],
     "titles": ["commercial manager", "quantity surveyor", "project manager"]},
]


def fails_mismatch(title, company):
    t = normalise_text(title)
    c = normalise_text(company)
    for rule in MISMATCH_RULES:
        if any(kw in c for kw in rule["companies"]) and any(kw in t for kw in rule["titles"]):
            return True
    return False


def make_bucket(score, stretch):
    if score >= 70 and stretch <= 5:
        return "A - Apply Now"
    if score >= 60 and stretch <= 7:
        return "B - High Upside"
    if score >= 50 and stretch <= 9:
        return "C - Network First"
    return "D - Skip"


# Pre-load the UK sponsor register once before scoring begins.
print("Loading UK visa sponsor register...")
load_uk_sponsor_register()

all_jobs = board_jobs + company_jobs + linkedin_jobs
removed_non_uk = 0
removed_mismatch = 0
removed_excluded = 0

for job in all_jobs:
    url = job.get("url", "")
    if not url or url in seen_urls:
        continue
    seen_urls.add(url)

    title = job.get("title", "")
    company = job.get("company", "")
    location = job.get("location", "")
    desc = job.get("desc", "")          # full text from the source, HTML already stripped
    source = job.get("source", "")

    description_available = job.get("description_available") or (
        "Yes" if desc and desc.strip() and desc.strip().lower() != title.strip().lower() else "No"
    )
    description_quality = describe_description_quality(
        desc, title, source, job.get("description_quality", "")
    )

    if location and not is_uk_loc(location):
        removed_non_uk += 1
        continue
    if fails_mismatch(title, company):
        removed_mismatch += 1
        continue

    total, track_points, reason, stretch, bucket = score_job(title, desc, location)

    if total == 0 and str(reason).upper().startswith("EXCLUDED"):
        removed_excluded += 1
        continue

    # Career pages and LinkedIn often return title-only rows, which starves the
    # content-based dimensions. The boost compensates for the missing text, so it
    # is largest exactly where the description is thinnest - and it is never
    # applied to a role that failed to map onto one of the candidate's tracks.
    if track_points > 0 and total >= 25:
        if source == "Career Page":
            boost = 20 if description_quality.startswith("Title Only") else 8
            total = min(total + boost, 100)
            reason += f" | +{boost} career-page boost"
        elif source == "LinkedIn":
            boost = 12 if description_quality.startswith("Title Only") else 5
            total = min(total + boost, 100)
            reason += f" | +{boost} LinkedIn limited-description boost"

    stretch = calculate_stretch(title, desc, company, total, track_points)
    bucket = make_bucket(total, stretch)

    country, continent = get_country_continent(location)
    if country in NON_UK_COUNTRIES:
        removed_non_uk += 1
        continue

    company_display = company
    if any(a in normalise_text(company_display) for a in AGENCIES) and "(Agency)" not in company_display:
        company_display += " (Agency)"

    description_main, description_overflow = split_description_for_excel(desc)

    risk_text = f"{title} {desc}"
    results.append({
        "Bucket": bucket,
        "Score /100": total,
        "Stretch (1-10)": stretch,
        "Title": title,
        "Company": company_display,
        "Location": location,
        "Country": country,
        "Continent": continent,
        "Salary": job.get("salary", "See listing"),
        "Posted": job.get("posted", ""),
        "Source": source,
        "Duplicate Sources": "",
        "Why It Matches": reason,
        "Job Description": description_main,
        "Job Description (continued)": description_overflow,
        "Description Available": description_available,
        "Description Quality": description_quality,
        "Sponsorship risk": sponsorship_risk_enhanced(risk_text, company_display),
        "Status": "To Review",
        "AI Remarks": "",
        "AI Review Decision": "",
        "Apply Link": url,
    })


# ── DEDUPLICATION: keep the best copy of a job, not the first one seen ───────
QUALITY_RANK = {"Full API": 3, "Board Summary": 2, "Title Only": 1, "Unavailable": 0}
SOURCE_RANK = {"Career Page": 3, "Adzuna": 2, "Reed": 2, "LinkedIn": 1}


def url_specificity(url):
    """A job-specific URL beats a generic careers landing page."""
    u = normalise_text(url)
    if not u:
        return 0
    if any(token in u for token in ["/job/", "/jobs/", "job_id", "jobid", "requisition", "/vacancy",
                                    "currentjobid", "/postings/", "viewjob", "-job-"]):
        return 3
    if u.rstrip("/").count("/") > 3:
        return 2
    return 1


def record_quality(row):
    """Ordering: completeness of the description, then URL specificity, then source."""
    quality_label = str(row.get("Description Quality", "")).split(" (")[0]
    return (
        QUALITY_RANK.get(quality_label, 1),
        len(row.get("Job Description", "") or "") + len(row.get("Job Description (continued)", "") or ""),
        url_specificity(row.get("Apply Link", "")),
        SOURCE_RANK.get(row.get("Source", ""), 1),
    )


best_by_key = {}
duplicate_sources = {}
for r in results:
    key = (normalise_text(r["Title"]), normalise_text(r["Company"]).replace(" (agency)", ""))
    duplicate_sources.setdefault(key, set()).add(r.get("Source", ""))
    incumbent = best_by_key.get(key)
    if incumbent is None:
        best_by_key[key] = r
        continue
    winner, loser = (r, incumbent) if record_quality(r) > record_quality(incumbent) else (incumbent, r)
    # Do not lose information the weaker copy had: fill any gaps from it.
    for field in ("Salary", "Posted", "Location", "Job Description", "Job Description (continued)",
                  "Apply Link", "Why It Matches"):
        if not str(winner.get(field, "") or "").strip() or str(winner.get(field, "")).strip() in ("See listing", ""):
            if str(loser.get(field, "") or "").strip():
                winner[field] = loser[field]
    winner["Score /100"] = max(winner.get("Score /100", 0), loser.get("Score /100", 0))
    best_by_key[key] = winner

deduped = []
for key, row in best_by_key.items():
    others = sorted(s for s in duplicate_sources.get(key, set()) if s and s != row.get("Source"))
    row["Duplicate Sources"] = ", ".join(others)
    deduped.append(row)

deduped.sort(key=lambda x: (x["Bucket"], -x["Score /100"], x["Stretch (1-10)"]))

# Production decision layer: deterministic rules rank every broadly collected
# candidate. The AI review cell is optional and is never called during this run.
deduped = append_rule_decision_layer(deduped)

a = [r for r in deduped if r["Bucket"] == "A - Apply Now"]
b = [r for r in deduped if r["Bucket"] == "B - High Upside"]
c = [r for r in deduped if r["Bucket"] == "C - Network First"]
print("=" * 55)
print(f"  Adzuna + Reed:       {len(board_jobs)}")
print(f"  Company pages:       {len(company_jobs)}")
print(f"  LinkedIn:            {len(linkedin_jobs)}")
print(f"  Removed non-UK:      {removed_non_uk}")
print(f"  Removed mismatches:  {removed_mismatch}")
print(f"  Removed excluded:    {removed_excluded}")
print(f"  Total unique:        {len(deduped)}")
print(f"  A - Apply Now:       {len(a)}")
print(f"  B - High Upside:     {len(b)}")
print(f"  C - Network First:   {len(c)}")
print("=" * 55)
print()
print("Bucket A - Apply Now:")
for i, r in enumerate(a, 1):
    print(f"  {i}. [{r['Score /100']}] Stretch:{r['Stretch (1-10)']} {r['Title']} @ {r['Company']} - {r['Location']}")

continent_counts = Counter(r["Continent"] for r in deduped)
country_counts = Counter(r["Country"] for r in deduped)
family_counts = Counter(r.get("Job Family", "Unknown") for r in deduped)
print("=" * 55)
print("Roles by Continent:")
for continent, count in sorted(continent_counts.items(), key=lambda x: -x[1]):
    print(f"  {continent:<20} {count}")
print()
print("Top Countries:")
for country, count in sorted(country_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {country:<20} {count}")
print()
print("Top Job Families:")
for family, count in sorted(family_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {family[:38]:<40} {count}")
print("=" * 55)

cols = [
    "Bucket", "Final Bucket", "Apply?", "Priority Score", "Overall Priority",
    "Score /100", "Stretch (1-10)", "Title", "Company", "Location", "Country", "Continent",
    "Salary", "Posted", "Source", "Duplicate Sources", "Description Available",
    "Description Quality", "Job Description", "Job Description (continued)",
    "Why It Matches", "Capability Scores", "Sponsorship risk", "Status", "AI Remarks",
    "AI Review Decision", "Apply Link", "Job Family", "Company Type", "Capability Fit",
    "Career Alignment", "Interview Probability Final", "Resume to Use", "Resume Changes Needed",
    "Resume Effort Required", "Cover Letter Effort Required", "Networking Recommendation",
    "Decision Reasoning",
]
df_final = pd.DataFrame(deduped)
if not df_final.empty:
    for col in cols:
        if col not in df_final.columns:
            df_final[col] = None
    df_final = df_final[cols]
else:
    df_final = pd.DataFrame(columns=cols)

networking_cols = [
    "Company", "Role", "Contact Name", "Contact Title", "LinkedIn URL", "Connection Sent",
    "Follow-up Due", "Response", "Referral Asked", "Notes", "Networking Recommendation", "Apply Link",
]
networking_rows = []
for r in deduped:
    if r["Bucket"] in ["A - Apply Now", "B - High Upside", "C - Network First"]:
        networking_rows.append({
            "Company": r["Company"], "Role": r["Title"],
            "Contact Name": "", "Contact Title": "", "LinkedIn URL": "",
            "Connection Sent": "No", "Follow-up Due": "", "Response": "",
            "Referral Asked": "No", "Notes": "",
            "Networking Recommendation": r.get("Networking Recommendation", ""),
            "Apply Link": r["Apply Link"],
        })
df_networking = pd.DataFrame(networking_rows, columns=networking_cols)

top20_cols = [
    "Overall Priority", "Company", "Title", "Job Family", "Capability Fit", "Career Alignment",
    "Interview Probability Final", "Resume to Use", "Resume Changes Needed",
    "Networking Recommendation", "Priority Score", "Apply?", "Final Bucket",
    "Decision Reasoning", "Apply Link",
]
# Sort deterministically by production priority score.
_top20_df = pd.DataFrame(deduped)
df_top20 = _top20_df.sort_values("Priority Score", ascending=False).head(20) if not _top20_df.empty else _top20_df
if not df_top20.empty:
    for col in top20_cols:
        if col not in df_top20.columns:
            df_top20[col] = None
    df_top20 = df_top20[top20_cols]
else:
    df_top20 = pd.DataFrame(columns=top20_cols)

# Employers worth a manual weekly check for this profile: their sites either
# block scraping or need their own search filters.
manual_checks = [
    {"Company": "Bank of England", "URL": "https://www.bankofengland.co.uk/careers", "Search": "data governance; regulatory data; data quality", "Action": "Manual weekly check"},
    {"Company": "FCA", "URL": "https://www.fca.org.uk/careers", "Search": "data governance; data quality; regulatory reporting", "Action": "Manual weekly check"},
    {"Company": "LSEG", "URL": "https://www.lseg.com/en/careers", "Search": "data governance; reference data; data quality", "Action": "Manual weekly check"},
    {"Company": "Moody's", "URL": "https://careers.moodys.com/", "Search": "data quality; data management; regulatory data", "Action": "Manual weekly check"},
    {"Company": "S&P Global", "URL": "https://careers.spglobal.com/", "Search": "data management; data governance; reporting analyst", "Action": "Manual weekly check"},
    {"Company": "Barclays", "URL": "https://search.jobs.barclays/search-jobs", "Search": "data governance; regulatory reporting; data quality", "Action": "Manual weekly check"},
    {"Company": "HSBC", "URL": "https://mycareer.hsbc.com/en_GB/external/SearchJobs", "Search": "data governance; regulatory reporting; data quality", "Action": "Manual weekly check"},
    {"Company": "NatWest", "URL": "https://jobs.natwestgroup.com/", "Search": "data analyst; data governance; risk data", "Action": "Manual weekly check"},
    {"Company": "Lloyds Banking Group", "URL": "https://www.lloydsbankinggroup.com/careers.html", "Search": "data quality; regulatory reporting; MI analyst", "Action": "Manual weekly check"},
    {"Company": "Deloitte", "URL": "https://apply.deloitte.com/careers/SearchJobs", "Search": "data governance; risk & regulatory data", "Action": "Referral route preferred"},
    {"Company": "EY", "URL": "https://careers.ey.com/ey/search/", "Search": "data governance; regulatory reporting", "Action": "Referral route preferred"},
    {"Company": "KPMG", "URL": "https://www.kpmgcareers.co.uk/", "Search": "data governance; regulatory reporting analyst", "Action": "Referral route preferred"},
    {"Company": "Experian", "URL": "https://www.experianplc.com/careers", "Search": "data quality; data governance; data management", "Action": "Manual weekly check"},
    {"Company": "Wells Fargo (UK)", "URL": "https://www.wellsfargojobs.com/en/jobs/", "Search": "data management; regulatory reporting", "Action": "Former employer - use internal network"},
]
df_manual_checks = pd.DataFrame(manual_checks)

output_file = OUTPUT_FILE
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_final.to_excel(writer, index=False, sheet_name="Jobs")
    df_top20.to_excel(writer, index=False, sheet_name="Rolling Top 20")
    df_networking.to_excel(writer, index=False, sheet_name="Networking Tracker")
    df_manual_checks.to_excel(writer, index=False, sheet_name="Manual Checks")

links_made = format_workbook(output_file)
print(f"Excel formatting applied: {links_made} clickable hyperlinks across Jobs, Rolling Top 20, Networking Tracker and Manual Checks.")

full_jd_rows = sum(1 for r in deduped if (r.get("Job Description") or "").strip())
print(f"Rows carrying a job description: {full_jd_rows}/{len(deduped)} (stored in full, untrimmed).")

files.download(output_file)
print(f"Downloaded {len(deduped)} roles to {output_file} with Jobs + Rolling Top 20 + Networking Tracker + Manual Checks tabs.")


In [ ]:
# OPTIONAL MANUAL AI REVIEW STEP - OPENROUTER
# Daily production ranking above does not call AI.
# To review a shortlist, set RUN_MANUAL_AI_REVIEW = True and run this cell after the main export cell.

RUN_MANUAL_AI_REVIEW = False
AI_REVIEW_LIMIT = 40       # Use 20-40 for practical review. Set to None to attempt all rows in batches.
AI_REVIEW_BATCH_SIZE = 10  # Batch calls reduce overhead versus one request per job.

# AI configuration in one place.
MODEL_NAME = "openai/gpt-4.1-mini"
MAX_RETRIES = 2
TEMPERATURE = 0.2
MAX_TOKENS = 2500

!pip install -U openai -q

import os
import json
import re
import time
import pandas as pd
from google.colab import files
from openai import OpenAI

# Reads the OpenRouter key from Colab Secrets first, then environment variables. Never hardcode keys.
def get_openrouter_api_key():
    try:
        from google.colab import userdata
        key = userdata.get("OPENROUTER_API_KEY")
        if key:
            return key
    except Exception:
        pass
    return os.environ.get("OPENROUTER_API_KEY", "")

# AI INPUT ONLY. This trimming exists to control OpenRouter token usage and is
# never applied to the Excel 'Job Description' column, which keeps the complete
# description written by the export cell.
def trim_job_description(description, max_words=1800):
    text = str(description or "").strip()
    if not text:
        return ""
    text = re.sub(r"\s+", " ", text)
    lower = text.lower()
    boilerplate_markers = [
        "equal opportunity", "diversity", "inclusion", "benefits", "privacy notice",
        "privacy policy", "legal notice", "about us", "about the company",
        "we are an equal", "reasonable accommodation", "background checks",
    ]
    cut_positions = [lower.find(marker) for marker in boilerplate_markers if lower.find(marker) > 400]
    if cut_positions:
        text = text[:min(cut_positions)].strip()

    words = text.split()
    if len(words) <= max_words:
        return text

    section_keywords = [
        "responsibilities", "requirements", "qualifications", "skills", "experience",
        "what you will do", "what you'll do", "about the role", "key responsibilities",
    ]
    sentences = re.split(r"(?<=[.!?])\s+", text)
    selected = []
    active = False
    for sentence in sentences:
        sentence_lower = sentence.lower()
        if any(keyword in sentence_lower for keyword in section_keywords):
            active = True
        if active:
            selected.append(sentence)
        if sum(len(s.split()) for s in selected) >= max_words:
            break

    if selected:
        return " ".join(selected).strip()
    return " ".join(words[:max_words]).strip()

# Safely extracts structured JSON arrays/objects from model responses.
def parse_ai_review_json(text):
    if not text:
        return []
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?", "", cleaned, flags=re.IGNORECASE).strip()
        cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        parsed = json.loads(cleaned)
    except Exception:
        match = re.search(r"\[.*\]|\{.*\}", cleaned, flags=re.DOTALL)
        if not match:
            return []
        try:
            parsed = json.loads(match.group(0))
        except Exception:
            return []
    if isinstance(parsed, dict):
        parsed = parsed.get("results", [])
    return parsed if isinstance(parsed, list) else []

# Builds a compact prompt payload with only fields the AI needs.
def compact_job_for_ai(row, row_id):
    return {
        "row_id": row_id,
        "job_title": row.get("Title", ""),
        "company": row.get("Company", ""),
        "location": row.get("Location", ""),
        "salary": row.get("Salary", ""),
        # Trimmed copy for the model only - row["Job Description"] is left untouched.
        "cleaned_job_description": trim_job_description(
            " ".join([row.get("Job Description", "") or "",
                      row.get("Job Description (continued)", "") or ""]).strip()
        ),
    }

# Provider abstraction so the rest of the notebook does not call OpenRouter directly.
class AIProvider:
    def __init__(self, api_key, model_name=MODEL_NAME, temperature=TEMPERATURE, max_tokens=MAX_TOKENS):
        self.client = OpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1")
        self.model_name = model_name
        self.temperature = temperature
        self.max_tokens = max_tokens

    def analyze_job(self, jobs, candidate_profile):
        prompt = f"""
You are reviewing jobs for this candidate:
{candidate_profile}

For each job, judge transferable fit based only on:
Job Title, Company, Location, Salary, Cleaned Job Description.

Return ONLY valid JSON array. No markdown. No commentary.
Each item must contain:
row_id, ai_review_decision, ai_remarks

Allowed ai_review_decision values: YES, MAYBE, NETWORK, NO.
ai_remarks must be one concise sentence explaining the decision.

Jobs:
{json.dumps(jobs, ensure_ascii=False)}
"""
        last_error = None
        for attempt in range(MAX_RETRIES):
            try:
                response = self.client.chat.completions.create(
                    model=self.model_name,
                    messages=[
                        {"role": "system", "content": "Return structured JSON only."},
                        {"role": "user", "content": prompt},
                    ],
                    temperature=self.temperature,
                    max_tokens=self.max_tokens,
                )
                content = response.choices[0].message.content
                return parse_ai_review_json(content)
            except Exception as e:
                last_error = e
                if attempt < MAX_RETRIES - 1:
                    time.sleep(2)
        print(f"OpenRouter batch failed after {MAX_RETRIES} attempt(s): {str(last_error)[:180]}")
        return []

# Built strictly from the CV. Do not add experience, seniority, salary
# expectations, visa status or achievements that are not listed here.
CANDIDATE_PROFILE = """
CANDIDATE: financial-services data professional, based in the United Kingdom.
Approximately 4 years of post-graduate experience (2021-2025). Analyst level.

EMPLOYMENT HISTORY (all of it):
1. Data Management Analyst, Wells Fargo (Feb 2024 - Jul 2025), Hyderabad.
   - Month-end reporting cycles; reconciliation of regulatory data and financial
     records across Commercial Banking portfolios.
   - Wrote and optimised complex SQL against large multi-source financial datasets
     to find and resolve data quality issues.
   - Maintained governance documentation, regulatory compliance records and
     reporting standards aligned to FR Y-14 requirements; documented 45
     audit-ready reports.
   - Managed FR Y-14 Schedule H1 and H2 submissions under Federal Reserve
     guidelines, producing bi-weekly, monthly and quarterly Data Quality packages.
   - Built and ran data quality checks for FR Y-14Q and BCBS 239 compliance.
   - Supported risk and compliance governance workstreams: project documentation,
     logs and audit trails.
2. Business Analyst, Mu Sigma Business Solutions (Jul 2021 - Feb 2024), Bengaluru.
   - Helped develop and implement a data governance framework: documentation
     standards, metadata management practices, information control procedures.
   - Managed metadata repositories, governance documentation and information
     standards across multiple systems and stakeholders.
   - Produced executive-level summaries from disparate departmental datasets.
   - Managed client relationships and check-ins on machine learning models for
     purchase-propensity audiences (stakeholder and delivery role, not model
     engineering); led planning sessions on variable prioritisation for deployment.
3. Intern, Tata Consultancy Services (Dec 2018 - Jan 2019), Kolkata.
   - Exposure to financial and regulatory data frameworks; documented systems and
     controls for client audits.

EDUCATION:
- MSc Programme and Project Management, University of Warwick (Sept 2025 - Sept 2026,
  in progress). Modules: project planning management and control, programme and
  project strategy, project financial management, project management in practice.
- BTech Information Technology, SRM Institute of Science and Technology (2017-2021).

TECHNICAL SKILLS (as listed on the CV): SQL, Python, Advanced Excel, VBA, Power BI,
Tableau, Alteryx, SharePoint, ServiceNow, JIRA, Trello, PowerPoint, Word.

DOMAIN AND FUNCTIONAL STRENGTHS: data governance, master data management, data
quality frameworks, data lineage, regulatory reporting (FR Y-14, FR Y-14Q, BCBS 239),
financial reconciliation, audit trails and document control, information management,
change management, business analysis, ad-hoc reporting, process improvement, senior
stakeholder reporting and cross-functional collaboration.

CERTIFICATIONS: Association for Project Management (APM), Machine Learning
certification, Data Analytics with Power BI. Spot awards at Mu Sigma and Wells Fargo.
PRINCE2 is listed as "familiar" only, not certified.

WHAT THIS CANDIDATE IS NOT (do not assume otherwise):
- Not a software, data or ML engineer; no production engineering experience.
- Not a data scientist or quantitative researcher.
- Not a qualified accountant, actuary, auditor or lawyer.
- No people-management or budget-ownership experience is stated.
- No stated salary expectation and no stated visa or right-to-work status.
  Never infer either.

HOW TO JUDGE FIT:
- Strong: data governance, data quality, data management, regulatory and risk
  reporting, financial data analysis, reconciliation and controls, MI/BI reporting,
  business analysis, and data-oriented project/PMO work at analyst, senior analyst,
  associate or specialist level in financial services or adjacent sectors.
- Possible: analytics, change/transformation analyst, data or analytics consulting,
  business process analysis, product or strategy analyst roles that lean on data.
- Weak: roles demanding engineering build skills, advanced statistical modelling,
  accounting qualifications, 8+ years of experience, or people-management scope.
- Judge transferable capability against the responsibilities in the advert. Do not
  require an exact title match, and do not reward a title just for containing
  "Analyst", "Data", "Business" or "Project".
"""

if not RUN_MANUAL_AI_REVIEW:
    print("Manual AI review is OFF. Set RUN_MANUAL_AI_REVIEW = True, add OPENROUTER_API_KEY in Colab Secrets, then run this cell to fill AI Remarks.")
else:
    api_key = get_openrouter_api_key()
    if not api_key:
        print("Missing OPENROUTER_API_KEY. Add it in Colab Secrets before running manual AI review.")
    else:
        provider = AIProvider(api_key=api_key)

        review_rows = list(deduped)
        review_rows = sorted(review_rows, key=lambda r: r.get("Priority Score", 0) or 0, reverse=True)
        if AI_REVIEW_LIMIT is not None:
            review_rows = review_rows[:AI_REVIEW_LIMIT]

        row_lookup = {}
        compact_rows = []
        for row_id, row in enumerate(review_rows, 1):
            row_lookup[row_id] = row
            compact_rows.append(compact_job_for_ai(row, row_id))

        print(f"Manual AI review via OpenRouter: {len(compact_rows)} jobs in batches of {AI_REVIEW_BATCH_SIZE}. Model: {MODEL_NAME}")
        for start in range(0, len(compact_rows), AI_REVIEW_BATCH_SIZE):
            batch = compact_rows[start:start + AI_REVIEW_BATCH_SIZE]
            reviews = provider.analyze_job(batch, CANDIDATE_PROFILE)
            for review in reviews:
                row = row_lookup.get(int(review.get("row_id", 0) or 0))
                if not row:
                    continue
                row["AI Remarks"] = review.get("ai_remarks", "")
                row["AI Review Decision"] = review.get("ai_review_decision", "")
            time.sleep(1)

        df_ai_jobs = pd.DataFrame(deduped)
        for col in cols:
            if col not in df_ai_jobs.columns:
                df_ai_jobs[col] = None
        df_ai_jobs = df_ai_jobs[cols]

        df_ai_top20 = df_ai_jobs.sort_values("Priority Score", ascending=False).head(20)
        df_ai_top20 = df_ai_top20[[c for c in top20_cols if c in df_ai_top20.columns]]
        df_ai_networking = pd.DataFrame(networking_rows, columns=networking_cols)
        df_ai_manual_checks = df_manual_checks.copy()

        output_file = AI_OUTPUT_FILE
        with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
            df_ai_jobs.to_excel(writer, index=False, sheet_name="Jobs")
            df_ai_top20.to_excel(writer, index=False, sheet_name="Rolling Top 20")
            df_ai_networking.to_excel(writer, index=False, sheet_name="Networking Tracker")
            df_ai_manual_checks.to_excel(writer, index=False, sheet_name="Manual Checks")

        ai_links = format_workbook(output_file)
        print(f"Excel formatting applied: {ai_links} clickable hyperlinks.")
        files.download(output_file)
        print(f"Downloaded AI-reviewed workbook: {output_file}")


In [ ]:
ubs_found = [r for r in deduped if "ubs" in r["Company"].lower()]
print(f"UBS roles in current results: {len(ubs_found)}")
for r in ubs_found:
    print(f"  [{r['Score /100']}] {r['Title']} @ {r['Company']} â€” {r['Location']} | Source: {r['Source']}")

In [ ]:
# Check if UBS came through LinkedIn before scoring
ubs_raw = [j for j in linkedin_jobs if "ubs" in j.get("company", "").lower()]
print(f"UBS in LinkedIn raw: {len(ubs_raw)}")
for j in ubs_raw:
    print(f"  {j['title']} â€” {j['location']}")

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import re

headers_ubs = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-GB,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Referer": "https://jobs.ubs.com",
}

UBS_URLS = [
    "https://jobs.ubs.com/TGnewUI/Search/home/HomeWithPreLoad?partnerid=25008&siteid=5012&PageType=searchResults&SearchType=linkquery&LinkID=15231",
    "https://jobs.ubs.com/TGnewUI/Search/home/HomeWithPreLoad?partnerid=25008&siteid=5012&PageType=JobListing&noback=1",
    "https://jobs.ubs.com/TGnewUI/Search/home/HomeWithPreLoad?partnerid=25008&siteid=5012&PageType=searchResults&SearchType=linkquery&LinkID=15231&keyWordSearch=data+governance&locationSearch=London",
    "https://jobs.ubs.com/TGnewUI/Search/home/HomeWithPreLoad?partnerid=25008&siteid=5012&PageType=searchResults&SearchType=linkquery&LinkID=15231&keyWordSearch=regulatory+reporting&locationSearch=London",
    "https://jobs.ubs.com/TGnewUI/Search/home/HomeWithPreLoad?partnerid=25008&siteid=5012&PageType=searchResults&SearchType=linkquery&LinkID=15231&keyWordSearch=data+quality&locationSearch=London",
]

# Also try their API endpoint
UBS_API_URLS = [
    "https://jobs.ubs.com/TGnewUI/Search/home/HomeWithPreLoad?partnerid=25008&siteid=5012&PageType=searchResults&SearchType=linkquery&LinkID=15231&keyWordSearch=data+governance+operations&locationSearch=London&format=json",
    "https://jobs.ubs.com/talentcommunity/api/v1/jobs?partnerid=25008&siteid=5012&keywords=data+governance&location=London",
    "https://jobs.ubs.com/api/jobs?keywords=data+governance&location=London&partnerid=25008",
]

RELEVANT = RELEVANT_TITLES  # candidate taxonomy from the settings cell
EXCLUDE = HARD_EXCLUDE      # context-aware filtering via title_is_excluded()

ubs_jobs = []

print("Testing UBS TGnewUI platform...")
print("-" * 55)

session = requests.Session()

for url in UBS_URLS:
    try:
        r = session.get(url, headers=headers_ubs, timeout=15)
        print(f"  {url[-80:]}")
        print(f"    Status: {r.status_code} | Size: {len(r.text)} chars")

        if r.status_code == 200 and len(r.text) > 1000:
            soup = BeautifulSoup(r.text, "html.parser")

            # Look for job listings in various formats
            found = 0

            # Method 1: Find job title links
            for a in soup.find_all("a", href=True):
                title = a.get_text(strip=True)
                href = a["href"]
                if len(title) < 8 or len(title) > 110: continue
                if not any(w in title.lower() for w in RELEVANT): continue
                if title_is_excluded(title): continue
                full_url = href if href.startswith("http") else f"https://jobs.ubs.com{href}"
                if any(j["url"] == full_url for j in ubs_jobs): continue
                ubs_jobs.append({
                    "title": title, "company": "UBS",
                    "location": "London, UK", "salary": "See listing",
                    "posted": "Live now", "desc": title,
                    "source": "Career Page", "url": full_url,
                })
                found += 1
                print(f"    âœ… Found: {title}")

            # Method 2: Look for JSON data in page
            scripts = soup.find_all("script")
            for script in scripts:
                if script.string and "jobTitle" in str(script.string):
                    print(f"    JSON data found in script tag â€” {len(script.string)} chars")
                    # Try to extract job titles
                    titles = re.findall(r'"jobTitle"\s*:\s*"([^"]+)"', script.string)
                    for title in titles:
                        if not any(w in title.lower() for w in RELEVANT): continue
                        if title_is_excluded(title): continue
                        print(f"    âœ… JSON job: {title}")

            # Method 3: Extract all text that looks like job titles
            all_text = soup.get_text(separator="\n")
            lines = [l.strip() for l in all_text.split("\n") if l.strip()]
            text_found = 0
            for line in lines:
                if len(line) < 10 or len(line) > 100: continue
                if not any(w in line.lower() for w in RELEVANT): continue
                if any(w in line.lower() for w in EXCLUDE): continue
                if any(nav in line.lower() for nav in ["cookie", "privacy", "sign in",
                    "log in", "search", "filter", "sort", "home", "about",
                    "contact", "careers at", "why ubs", "our culture"]): continue
                if text_found < 20:
                    print(f"    Text: {line}")
                text_found += 1

    except Exception as e:
        print(f"  Error: {e}")
    time.sleep(1)

# Try API endpoints
print()
print("Testing UBS API endpoints...")
print("-" * 55)
for url in UBS_API_URLS:
    try:
        r = session.get(url, headers={**headers_ubs, "Accept": "application/json"}, timeout=12)
        print(f"  Status: {r.status_code} | Size: {len(r.text)} | URL: {url[-60:]}")
        if r.status_code == 200:
            print(f"  Preview: {r.text[:300]}")
    except Exception as e:
        print(f"  Error: {e}")

print()
print(f"UBS jobs found: {len(ubs_jobs)}")

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import json

headers_ubs = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-GB,en;q=0.9",
}

session = requests.Session()
url = "https://jobs.ubs.com/TGnewUI/Search/home/HomeWithPreLoad?partnerid=25008&siteid=5012&PageType=searchResults&SearchType=linkquery&LinkID=15231"

r = session.get(url, headers=headers_ubs, timeout=20)
soup = BeautifulSoup(r.text, "html.parser")

print(f"Page size: {len(r.text)} chars")
print()

# Extract all script tags and look for job data
scripts = soup.find_all("script")
print(f"Total script tags: {len(scripts)}")

for i, script in enumerate(scripts):
    if not script.string:
        continue
    s = script.string

    # Look for job-related JSON keys
    if any(key in s for key in ["jobTitle", "JobTitle", "job_title", "position",
                                  "requisition", "Requisition", "openings"]):
        print(f"\n=== Script {i} ({len(s)} chars) ===")
        print(f"Preview: {s[:500]}")
        print()

        # Try to find all job title patterns
        patterns = [
            r'"jobTitle"\s*:\s*"([^"]+)"',
            r'"JobTitle"\s*:\s*"([^"]+)"',
            r'"title"\s*:\s*"([^"]+)"',
            r'"Title"\s*:\s*"([^"]+)"',
            r'"name"\s*:\s*"([^"]+)"',
            r'"position"\s*:\s*"([^"]+)"',
            r'"RequisitionTitle"\s*:\s*"([^"]+)"',
        ]

        for pattern in patterns:
            matches = re.findall(pattern, s)
            if matches:
                print(f"Pattern '{pattern[:30]}' found {len(matches)} matches:")
                for m in matches[:10]:
                    print(f"  - {m}")

        # Try to parse as JSON
        json_matches = re.findall(r'\{[^{}]{100,}\}', s)
        for jm in json_matches[:3]:
            try:
                data = json.loads(jm)
                print(f"Valid JSON object found with keys: {list(data.keys())[:10]}")
            except:
                pass